# Vibration Anomaly Detection -- Kaggle test run (CPU, offline)

Self-contained snapshot of the `vibration_anomaly_detection` project (regenerated by `scripts/build_kaggle_notebook.py` -- not live-synced, re-run that script and re-push if the source changes). Reconstructs the `vibration_ad` package on disk from embedded source, converts the CWRU 12kHz Drive-End + Normal-baseline data from the attached `esraakhaled299/cwru-data` input dataset (.mat -> our .npz convention), then runs the full train -> evaluate -> infer pipeline on CPU.

No Internet needed -- the attached input dataset is readable regardless of the kernel's Internet setting (this Kaggle account isn't phone-verified, which disables Internet/GPU in kernels outright; see notes/07_dvc_and_kaggle.md). GPU is deliberately not requested either -- CPU is fine, matching the local machine this also needs to run on eventually.

See the parent project's `notes/` for why the pipeline is built this way (STFT choice, split strategy, threshold selection, etc.) -- this notebook is deliberately just the runner, not re-documented here.

## 1. Reconstruct the `vibration_ad` package + config on disk

In [ ]:
import base64, pathlib

WORKDIR = pathlib.Path('/kaggle/working')
SRC_FILES = {
    'src/vibration_ad/__init__.py': (
        'IiIiVW5zdXBlcnZpc2VkIHZpYnJhdGlvbiBhbm9tYWx5IGRldGVjdGlvbiBvbiB0aGUgQ1dSVSBiZWFyaW5nIGRhdGFzZXQuIiIiCgpfX3ZlcnNpb25fXyA9ICIwLjEuMCIK'
    ),
    'src/vibration_ad/config.py': (
        'IiIiTG9hZCB0aGUgWUFNTCBjb25maWcgaW50byBhIHNtYWxsIHRyZWUgb2YgZGF0YWNsYXNzZXMuCgpFdmVyeXRoaW5nIGRvd25zdHJlYW0gKGRhdGEgcHJlcCwgdHJhaW5pbmcsIGV2YWx1YXRpb24sIGluZmVyZW5jZSkgdGFrZXMKb25lIG9mIHRoZXNlIGBDb25maWdgIG9iamVjdHMgaW5zdGVhZCBvZiBzY2F0dGVyZWQgQ0xJIGZsYWdzLCBzbyBhIHJ1biBpcwpmdWxseSBkZXNjcmliZWQgYnkgYSBzaW5nbGUgWUFNTCBmaWxlIChzZWUgY29uZmlncy9kZWZhdWx0LnlhbWwpLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB5YW1sCgoKQGRhdGFjbGFzcwpjbGFzcyBQYXRoc0NvbmZpZzoKICAgIHJhd19kaXI6IHN0cgogICAgcHJvY2Vzc2VkX2Rpcjogc3RyCiAgICBtb2RlbHNfZGlyOiBzdHIKICAgIGZpZ3VyZXNfZGlyOiBzdHIKICAgIHJlcG9ydHNfZGlyOiBzdHIKCgpAZGF0YWNsYXNzCmNsYXNzIERhdGFDb25maWc6CiAgICBzZW5zb3I6IHN0cgogICAgZnNfaHo6IGludAogICAgdHJhaW5fcnBtczogbGlzdFtpbnRdCiAgICBob2xkb3V0X3JwbTogaW50CiAgICBmYXVsdF90eXBlczogbGlzdFtzdHJdCiAgICBmYXVsdF9ycG1zOiBsaXN0W2ludF0KCgpAZGF0YWNsYXNzCmNsYXNzIFdpbmRvd2luZ0NvbmZpZzoKICAgIHNlZ21lbnRfbGVuZ3RoOiBpbnQKICAgIHRyYWluX3N0cmlkZTogaW50CiAgICBldmFsX3N0cmlkZTogaW50CgoKQGRhdGFjbGFzcwpjbGFzcyBTVEZUQ29uZmlnOgogICAgbnBlcnNlZzogaW50CiAgICBob3BfbGVuZ3RoOiBpbnQKICAgIGxvZ19lcHNpbG9uOiBmbG9hdAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG5vdmVybGFwKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5ucGVyc2VnIC0gc2VsZi5ob3BfbGVuZ3RoCgoKQGRhdGFjbGFzcwpjbGFzcyBNb2RlbENvbmZpZzoKICAgIGxhdGVudF9kaW06IGludAogICAgYmFzZV9jaGFubmVsczogaW50CgoKQGRhdGFjbGFzcwpjbGFzcyBUcmFpbkNvbmZpZzoKICAgIGJhdGNoX3NpemU6IGludAogICAgZXBvY2hzOiBpbnQKICAgIGxlYXJuaW5nX3JhdGU6IGZsb2F0CiAgICB3ZWlnaHRfZGVjYXk6IGZsb2F0CiAgICBlYXJseV9zdG9wcGluZ19wYXRpZW5jZTogaW50CiAgICB2YWxfZnJhY3Rpb25fb2ZfaG9sZG91dDogZmxvYXQKICAgIGRldmljZTogc3RyCgoKQGRhdGFjbGFzcwpjbGFzcyBFdmFsdWF0aW9uQ29uZmlnOgogICAgdGhyZXNob2xkX3BlcmNlbnRpbGU6IGZsb2F0CgoKQGRhdGFjbGFzcwpjbGFzcyBDb25maWc6CiAgICBzZWVkOiBpbnQKICAgIHBhdGhzOiBQYXRoc0NvbmZpZwogICAgZGF0YTogRGF0YUNvbmZpZwogICAgd2luZG93aW5nOiBXaW5kb3dpbmdDb25maWcKICAgIHN0ZnQ6IFNURlRDb25maWcKICAgIG1vZGVsOiBNb2RlbENvbmZpZwogICAgdHJhaW46IFRyYWluQ29uZmlnCiAgICBldmFsdWF0aW9uOiBFdmFsdWF0aW9uQ29uZmlnCiAgICAjIERpcmVjdG9yeSB0aGUgY29uZmlnIGZpbGUgbGl2ZXMgaW47IHJlbGF0aXZlIHBhdGhzIGluIGBwYXRoc2AgYXJlCiAgICAjIHJlc29sdmVkIGFnYWluc3QgdGhlIHByb2plY3Qgcm9vdCAocGFyZW50IG9mIGNvbmZpZ3MvKSwgbm90IGFnYWluc3QgY3dkLgogICAgcHJvamVjdF9yb290OiBQYXRoID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PVBhdGguY3dkKQoKICAgIGRlZiByZXNvbHZlKHNlbGYsIHJlbGF0aXZlX3BhdGg6IHN0cikgLT4gUGF0aDoKICAgICAgICByZXR1cm4gKHNlbGYucHJvamVjdF9yb290IC8gcmVsYXRpdmVfcGF0aCkucmVzb2x2ZSgpCgoKZGVmIGxvYWRfY29uZmlnKGNvbmZpZ19wYXRoOiBzdHIgfCBQYXRoKSAtPiBDb25maWc6CiAgICBjb25maWdfcGF0aCA9IFBhdGgoY29uZmlnX3BhdGgpLnJlc29sdmUoKQogICAgd2l0aCBvcGVuKGNvbmZpZ19wYXRoKSBhcyBmOgogICAgICAgIHJhdyA9IHlhbWwuc2FmZV9sb2FkKGYpCgogICAgcHJvamVjdF9yb290ID0gY29uZmlnX3BhdGgucGFyZW50LnBhcmVudCAgIyBjb25maWdzL2RlZmF1bHQueWFtbCAtPiBwcm9qZWN0IHJvb3QKCiAgICByZXR1cm4gQ29uZmlnKAogICAgICAgIHNlZWQ9cmF3WyJzZWVkIl0sCiAgICAgICAgcGF0aHM9UGF0aHNDb25maWcoKipyYXdbInBhdGhzIl0pLAogICAgICAgIGRhdGE9RGF0YUNvbmZpZygqKnJhd1siZGF0YSJdKSwKICAgICAgICB3aW5kb3dpbmc9V2luZG93aW5nQ29uZmlnKCoqcmF3WyJ3aW5kb3dpbmciXSksCiAgICAgICAgc3RmdD1TVEZUQ29uZmlnKCoqcmF3WyJzdGZ0Il0pLAogICAgICAgIG1vZGVsPU1vZGVsQ29uZmlnKCoqcmF3WyJtb2RlbCJdKSwKICAgICAgICB0cmFpbj1UcmFpbkNvbmZpZygqKnJhd1sidHJhaW4iXSksCiAgICAgICAgZXZhbHVhdGlvbj1FdmFsdWF0aW9uQ29uZmlnKCoqcmF3WyJldmFsdWF0aW9uIl0pLAogICAgICAgIHByb2plY3Rfcm9vdD1wcm9qZWN0X3Jvb3QsCiAgICApCg=='
    ),
    'src/vibration_ad/data/__init__.py': (
        ''
    ),
    'src/vibration_ad/data/catalog.py': (
        'IiIiUGFyc2UgQ1dSVSAubnB6IGZpbGVuYW1lcyBpbnRvIHN0cnVjdHVyZWQgbWV0YWRhdGEuCgpDV1JVX0JlYXJpbmdfTnVtUHkgbmFtaW5nIGNvbnZlbnRpb24gKHNlZSBkYXRhL1JFQURNRS5tZCk6CiAgICB7cnBtfV9Ob3JtYWwubnB6CiAgICB7cnBtfV97ZmF1bHR9X3tzaXplX21pbHN9X3tlbmR9Lm5wegoKYGZhdWx0YCBpcyBvbmUgb2YgSVIgKGlubmVyIHJhY2UpLCBCIChiYWxsKSwgT1JAMyAvIE9SQDYgLyBPUkAxMiAob3V0ZXIKcmFjZSwgY2xvY2sgcG9zaXRpb24gcmVsYXRpdmUgdG8gdGhlIGxvYWQgem9uZSkuIGBzaXplX21pbHNgIGlzIHRoZSBmYXVsdApkaWFtZXRlciBpbiB0aG91c2FuZHRocyBvZiBhbiBpbmNoLiBgZW5kYCBpcyBERTEyL0RFNDgvRkU7IHdlIG9ubHkgdXNlCkRFMTIgKDEyIGtIeiBEcml2ZS1FbmQpIHNvIGl0J3MgY29uc3RhbnQgYWNyb3NzIG91ciByYXcgZmlsZXMuCgpOb3RlOiB0aGVzZSBmYXVsdCBsYWJlbHMgYXJlIE9OTFkgZXZlciB1c2VkIGZvciBldmFsdWF0aW9uIGluIHRoaXMgcHJvamVjdAooY29tcHV0aW5nIFJPQy1BVUMsIGNvbmZ1c2lvbiBtYXRyaWNlcywgcGVyLXNldmVyaXR5IGRldGVjdGlvbiByYXRlKS4gVGhlCmF1dG9lbmNvZGVyIGl0c2VsZiBpcyB0cmFpbmVkIGV4Y2x1c2l2ZWx5IG9uIHdpbmRvd3MgZnJvbSBOb3JtYWwubnB6IGZpbGVzCmFuZCBuZXZlciBzZWVzIGEgbGFiZWwuIFNlZSBub3Rlcy8wNF9ldmFsdWF0aW9uX25vdGVzLm1kLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKTk9STUFMX1JFID0gcmUuY29tcGlsZShyIl4oP1A8cnBtPlxkKylfTm9ybWFsXC5ucHokIikKRkFVTFRfUkUgPSByZS5jb21waWxlKAogICAgciJeKD9QPHJwbT5cZCspXyg/UDxmYXVsdD5JUnxCfE9SQDN8T1JANnxPUkAxMilfKD9QPHNpemU+XGQrKV8oP1A8ZW5kPkRFXGQrfEZFKVwubnB6JCIKKQoKCmRlZiBwYXJzZV9maWxlbmFtZShmaWxlbmFtZTogc3RyKSAtPiBkaWN0OgogICAgIiIiUmV0dXJuIG1ldGFkYXRhIGZvciBvbmUgQ1dSVSBucHogZmlsZW5hbWUsIG9yIHJhaXNlIFZhbHVlRXJyb3IuIiIiCiAgICBtID0gTk9STUFMX1JFLm1hdGNoKGZpbGVuYW1lKQogICAgaWYgbToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAicnBtIjogaW50KG0uZ3JvdXAoInJwbSIpKSwKICAgICAgICAgICAgImZhdWx0X3R5cGUiOiAiTm9ybWFsIiwKICAgICAgICAgICAgImZhdWx0X3NpemVfbWlscyI6IDAsCiAgICAgICAgICAgICJlbmQiOiBOb25lLAogICAgICAgICAgICAibGFiZWwiOiAwLAogICAgICAgIH0KCiAgICBtID0gRkFVTFRfUkUubWF0Y2goZmlsZW5hbWUpCiAgICBpZiBtOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJycG0iOiBpbnQobS5ncm91cCgicnBtIikpLAogICAgICAgICAgICAiZmF1bHRfdHlwZSI6IG0uZ3JvdXAoImZhdWx0IiksCiAgICAgICAgICAgICJmYXVsdF9zaXplX21pbHMiOiBpbnQobS5ncm91cCgic2l6ZSIpKSwKICAgICAgICAgICAgImVuZCI6IG0uZ3JvdXAoImVuZCIpLAogICAgICAgICAgICAibGFiZWwiOiAxLAogICAgICAgIH0KCiAgICByYWlzZSBWYWx1ZUVycm9yKGYiRmlsZW5hbWUgZG9lc24ndCBtYXRjaCBDV1JVIG5hbWluZyBjb252ZW50aW9uOiB7ZmlsZW5hbWV9IikKCgpkZWYgYnVpbGRfY2F0YWxvZyhyYXdfZGlyOiBQYXRoKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJTY2FuIHJhd19kaXIgZm9yIC5ucHogZmlsZXMgYW5kIHJldHVybiBvbmUgbWV0YWRhdGEgcm93IHBlciBmaWxlLiIiIgogICAgcm93cyA9IFtdCiAgICBmb3IgcGF0aCBpbiBzb3J0ZWQocmF3X2Rpci5nbG9iKCIqLm5weiIpKToKICAgICAgICBtZXRhID0gcGFyc2VfZmlsZW5hbWUocGF0aC5uYW1lKQogICAgICAgIG1ldGFbImZpbGVuYW1lIl0gPSBwYXRoLm5hbWUKICAgICAgICBtZXRhWyJmaWxlcGF0aCJdID0gc3RyKHBhdGgpCiAgICAgICAgcm93cy5hcHBlbmQobWV0YSkKCiAgICBpZiBub3Qgcm93czoKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJObyAubnB6IGZpbGVzIGZvdW5kIGluIHtyYXdfZGlyfS4gUnVuICIKICAgICAgICAgICAgImBweXRob24gLW0gdmlicmF0aW9uX2FkLmRhdGEuZG93bmxvYWRgIGZpcnN0LiIKICAgICAgICApCgogICAgZGYgPSBwZC5EYXRhRnJhbWUocm93cykKICAgIGNvbHMgPSBbImZpbGVuYW1lIiwgImZpbGVwYXRoIiwgInJwbSIsICJmYXVsdF90eXBlIiwgImZhdWx0X3NpemVfbWlscyIsICJlbmQiLCAibGFiZWwiXQogICAgcmV0dXJuIGRmW2NvbHNdLnNvcnRfdmFsdWVzKFsicnBtIiwgImZhdWx0X3R5cGUiLCAiZmF1bHRfc2l6ZV9taWxzIl0pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKCgpkZWYgc2F2ZV9jYXRhbG9nKGRmOiBwZC5EYXRhRnJhbWUsIHByb2Nlc3NlZF9kaXI6IFBhdGgpIC0+IFBhdGg6CiAgICBwcm9jZXNzZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dF9wYXRoID0gcHJvY2Vzc2VkX2RpciAvICJjYXRhbG9nLmNzdiIKICAgIGRmLnRvX2NzdihvdXRfcGF0aCwgaW5kZXg9RmFsc2UpCiAgICByZXR1cm4gb3V0X3BhdGgK'
    ),
    'src/vibration_ad/data/dataset.py': (
        'IiIiQnVpbGQgdGhlIHRyYWluL3ZhbC90ZXN0IHNwZWN0cm9ncmFtIGFycmF5cyB1c2VkIGJ5IGV2ZXJ5IG90aGVyIHN0YWdlLgoKU3BsaXQgc3RyYXRlZ3kgKHNlZSBub3Rlcy8wMV9kYXRhc2V0X25vdGVzLm1kIGFuZCBub3Rlcy8wNF9ldmFsdWF0aW9uX25vdGVzLm1kCmZvciB0aGUgZnVsbCByZWFzb25pbmcpOgoKICBUUkFJTiAoaGVhbHRoeSBvbmx5LCBkZW5zZSB3aW5kb3cgb3ZlcmxhcCk6CiAgICBOb3JtYWwubnB6IHdpbmRvd3MgZnJvbSBjb25maWcuZGF0YS50cmFpbl9ycG1zIChkZWZhdWx0OiAxNzMwLzE3NTAvMTc3MikuCgogIFZBTCAoaGVhbHRoeSBvbmx5LCBzcGFyc2Ugb3ZlcmxhcCk6CiAgICBmaXJzdCBoYWxmIG9mIE5vcm1hbC5ucHogd2luZG93cyBmcm9tIGNvbmZpZy5kYXRhLmhvbGRvdXRfcnBtICgxNzk3KS4KICAgIFVzZWQgZm9yIGVhcmx5IHN0b3BwaW5nIGFuZCBmb3IgcGlja2luZyB0aGUgYW5vbWFseSB0aHJlc2hvbGQuCgogIFRFU1QgKGhlYWx0aHkgKyBmYXVsdHksIHNwYXJzZSBvdmVybGFwKToKICAgIHNlY29uZCBoYWxmIG9mIE5vcm1hbC5ucHogd2luZG93cyBmcm9tIGhvbGRvdXRfcnBtICh0aGUgIm5vcm1hbCIgY2xhc3MpCiAgICBQTFVTIGV2ZXJ5IGZhdWx0IGZpbGUgZnJvbSBjb25maWcuZGF0YS5mYXVsdF9ycG1zIHggZmF1bHRfdHlwZXMgKHRoZQogICAgImFub21hbHkiIGNsYXNzKS4gTm8gZmF1bHQgZGF0YSwgYW5kIG5vIGhvbGRvdXQtUlBNIGhlYWx0aHkgZGF0YSwgaXMKICAgIGV2ZXIgc2VlbiBkdXJpbmcgdHJhaW5pbmcuCgpUaGUgdmFsL3Rlc3Qgc3BsaXQgb2YgdGhlIGhvbGRvdXQgaGVhbHRoeSBmaWxlIGlzIGEgY29udGlndW91cyBzcGxpdCAoZmlyc3QKTiB3aW5kb3dzIHZzLiBsYXN0IE0gd2luZG93cyksIG5vdCBhIHJhbmRvbSBzaHVmZmxlOiBiZWNhdXNlIHdpbmRvd3Mgb3ZlcmxhcApieSBkZXNpZ24sIGEgcmFuZG9tIHNwbGl0IHdvdWxkIHB1dCBuZWFyLWR1cGxpY2F0ZSB3aW5kb3dzIG9uIGJvdGggc2lkZXMgb2YKdGhlIHNwbGl0IGFuZCBxdWlldGx5IGxlYWsgaW5mb3JtYXRpb24gZnJvbSB2YWwgaW50byB0ZXN0LgoKVGhpcyBtb2R1bGUgb25seSBkb2VzIGRldGVybWluaXN0aWMgZGF0YSBlbmdpbmVlcmluZyAobm8gbGVhcm5lZCBzdGF0ZSAtLQpub3JtYWxpemF0aW9uIGlzIGZpdCBsYXRlciwgaW4gdHJhaW4ucHkpLiBSZXN1bHRzIGFyZSBjYWNoZWQgdG8KZGF0YS9wcm9jZXNzZWQvIGFzIHBsYWluIC5ucHkvLmNzdiBzbyByZXBlYXRlZCBydW5zIHNraXAgdGhlIFNURlQgcmVjb21wdXRlLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSB2aWJyYXRpb25fYWQuY29uZmlnIGltcG9ydCBDb25maWcKZnJvbSB2aWJyYXRpb25fYWQuZGF0YS5jYXRhbG9nIGltcG9ydCBidWlsZF9jYXRhbG9nLCBzYXZlX2NhdGFsb2cKZnJvbSB2aWJyYXRpb25fYWQuZGF0YS53aW5kb3dzIGltcG9ydCBtYWtlX3dpbmRvd3MKZnJvbSB2aWJyYXRpb25fYWQuZmVhdHVyZXMuc3BlY3Ryb2dyYW0gaW1wb3J0IHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcwoKCmRlZiBsb2FkX3NpZ25hbChucHpfcGF0aDogc3RyLCBzZW5zb3Jfa2V5OiBzdHIpIC0+IG5wLm5kYXJyYXk6CiAgICB3aXRoIG5wLmxvYWQobnB6X3BhdGgpIGFzIGRhdGE6CiAgICAgICAgaWYgc2Vuc29yX2tleSBub3QgaW4gZGF0YToKICAgICAgICAgICAgcmFpc2UgS2V5RXJyb3IoZiJ7bnB6X3BhdGh9IGhhcyBubyAne3NlbnNvcl9rZXl9JyBhcnJheSAoaGFzOiB7bGlzdChkYXRhLmtleXMoKSl9KSIpCiAgICAgICAgcmV0dXJuIGRhdGFbc2Vuc29yX2tleV0uYXN0eXBlKG5wLmZsb2F0NjQpLnJhdmVsKCkKCgpkZWYgX3dpbmRvd3NfZm9yX2ZpbGUoZmlsZXBhdGg6IHN0ciwgc2Vuc29yOiBzdHIsIHNlZ21lbnRfbGVuZ3RoOiBpbnQsIHN0cmlkZTogaW50KSAtPiBucC5uZGFycmF5OgogICAgc2lnbmFsID0gbG9hZF9zaWduYWwoZmlsZXBhdGgsIHNlbnNvcikKICAgIHJldHVybiBtYWtlX3dpbmRvd3Moc2lnbmFsLCBzZWdtZW50X2xlbmd0aCwgc3RyaWRlKQoKCmRlZiBfc3BlY3Ryb2dyYW1zX2Zvcl9yb3dzKAogICAgcm93czogcGQuRGF0YUZyYW1lLCBjb25maWc6IENvbmZpZywgc3RyaWRlOiBpbnQKKSAtPiB0dXBsZVtucC5uZGFycmF5LCBwZC5EYXRhRnJhbWVdOgogICAgIiIiV2luZG93ICsgU1RGVCBldmVyeSBmaWxlIGluIGByb3dzYCwgY29uY2F0ZW5hdGUsIGFuZCByZXR1cm4gbWF0Y2hpbmcKICAgIHBlci13aW5kb3cgbWV0YWRhdGEgKG9uZSBtZXRhZGF0YSByb3cgcGVyIG91dHB1dCB3aW5kb3csIG5vdCBwZXIgZmlsZSkuCiAgICAiIiIKICAgIGFsbF93aW5kb3dzID0gW10KICAgIG1ldGFfcm93cyA9IFtdCiAgICBmb3IgXywgcm93IGluIHJvd3MuaXRlcnJvd3MoKToKICAgICAgICB3ID0gX3dpbmRvd3NfZm9yX2ZpbGUocm93WyJmaWxlcGF0aCJdLCBjb25maWcuZGF0YS5zZW5zb3IsIGNvbmZpZy53aW5kb3dpbmcuc2VnbWVudF9sZW5ndGgsIHN0cmlkZSkKICAgICAgICBpZiBsZW4odykgPT0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBhbGxfd2luZG93cy5hcHBlbmQodykKICAgICAgICBmb3IgXyBpbiByYW5nZShsZW4odykpOgogICAgICAgICAgICBtZXRhX3Jvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgewogICAgICAgICAgICAgICAgICAgICJycG0iOiByb3dbInJwbSJdLAogICAgICAgICAgICAgICAgICAgICJmYXVsdF90eXBlIjogcm93WyJmYXVsdF90eXBlIl0sCiAgICAgICAgICAgICAgICAgICAgImZhdWx0X3NpemVfbWlscyI6IHJvd1siZmF1bHRfc2l6ZV9taWxzIl0sCiAgICAgICAgICAgICAgICAgICAgImxhYmVsIjogcm93WyJsYWJlbCJdLAogICAgICAgICAgICAgICAgICAgICJzb3VyY2VfZmlsZSI6IHJvd1siZmlsZW5hbWUiXSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKQoKICAgIHdpbmRvd3MgPSBucC5jb25jYXRlbmF0ZShhbGxfd2luZG93cywgYXhpcz0wKQogICAgc3BlY3Ryb2dyYW1zID0gd2luZG93c190b19sb2dfc3BlY3Ryb2dyYW1zKHdpbmRvd3MsIGNvbmZpZy5kYXRhLmZzX2h6LCBjb25maWcuc3RmdCkKICAgIHJldHVybiBzcGVjdHJvZ3JhbXMsIHBkLkRhdGFGcmFtZShtZXRhX3Jvd3MpCgoKZGVmIF9jYWNoZV9wYXRocyhwcm9jZXNzZWRfZGlyOiBQYXRoKSAtPiBkaWN0W3N0ciwgUGF0aF06CiAgICByZXR1cm4gewogICAgICAgICJ0cmFpbl9YIjogcHJvY2Vzc2VkX2RpciAvICJ0cmFpbl9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidmFsX1giOiBwcm9jZXNzZWRfZGlyIC8gInZhbF9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidmFsX21ldGEiOiBwcm9jZXNzZWRfZGlyIC8gInZhbF9tZXRhLmNzdiIsCiAgICAgICAgInRlc3RfWCI6IHByb2Nlc3NlZF9kaXIgLyAidGVzdF9zcGVjdHJvZ3JhbXMubnB5IiwKICAgICAgICAidGVzdF9tZXRhIjogcHJvY2Vzc2VkX2RpciAvICJ0ZXN0X21ldGEuY3N2IiwKICAgIH0KCgpkZWYgYnVpbGRfcmF3X2RhdGFzZXRzKAogICAgY29uZmlnOiBDb25maWcsIGZvcmNlOiBib29sID0gRmFsc2UKKSAtPiB0dXBsZVtucC5uZGFycmF5LCBucC5uZGFycmF5LCBwZC5EYXRhRnJhbWUsIG5wLm5kYXJyYXksIHBkLkRhdGFGcmFtZV06CiAgICAiIiJSZXR1cm5zICh0cmFpbl9YLCB2YWxfWCwgdmFsX21ldGEsIHRlc3RfWCwgdGVzdF9tZXRhKS4KCiAgICB0cmFpbl9YOiAoTl90cmFpbiwgRiwgVCkgZmxvYXQzMiwgdW4tbm9ybWFsaXplZCBsb2ctbWFnbml0dWRlIHNwZWN0cm9ncmFtcy4KICAgIHZhbF9YOiAgIChOX3ZhbCwgRiwgVCksIGhlYWx0aHkgb25seS4KICAgIHRlc3RfWDogIChOX3Rlc3QsIEYsIFQpLCBoZWFsdGh5IChsYWJlbD0wKSArIGZhdWx0eSAobGFiZWw9MSkuCiAgICAqX21ldGE6ICBvbmUgcm93IHBlciB3aW5kb3csIGNvbHVtbnMgcnBtL2ZhdWx0X3R5cGUvZmF1bHRfc2l6ZV9taWxzL2xhYmVsL3NvdXJjZV9maWxlLgogICAgIiIiCiAgICBwcm9jZXNzZWRfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnByb2Nlc3NlZF9kaXIpCiAgICBwcm9jZXNzZWRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGhzID0gX2NhY2hlX3BhdGhzKHByb2Nlc3NlZF9kaXIpCgogICAgaWYgbm90IGZvcmNlIGFuZCBhbGwocC5leGlzdHMoKSBmb3IgcCBpbiBwYXRocy52YWx1ZXMoKSk6CiAgICAgICAgdHJhaW5fWCA9IG5wLmxvYWQocGF0aHNbInRyYWluX1giXSkKICAgICAgICB2YWxfWCA9IG5wLmxvYWQocGF0aHNbInZhbF9YIl0pCiAgICAgICAgdmFsX21ldGEgPSBwZC5yZWFkX2NzdihwYXRoc1sidmFsX21ldGEiXSkKICAgICAgICB0ZXN0X1ggPSBucC5sb2FkKHBhdGhzWyJ0ZXN0X1giXSkKICAgICAgICB0ZXN0X21ldGEgPSBwZC5yZWFkX2NzdihwYXRoc1sidGVzdF9tZXRhIl0pCiAgICAgICAgcmV0dXJuIHRyYWluX1gsIHZhbF9YLCB2YWxfbWV0YSwgdGVzdF9YLCB0ZXN0X21ldGEKCiAgICByYXdfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnJhd19kaXIpCiAgICBjYXRhbG9nID0gYnVpbGRfY2F0YWxvZyhyYXdfZGlyKQogICAgc2F2ZV9jYXRhbG9nKGNhdGFsb2csIHByb2Nlc3NlZF9kaXIpCgogICAgIyAtLS0gVFJBSU46IE5vcm1hbCBmaWxlcyBmcm9tIHRyYWluX3JwbXMsIGRlbnNlIG92ZXJsYXAgLS0tCiAgICB0cmFpbl9yb3dzID0gY2F0YWxvZ1sKICAgICAgICAoY2F0YWxvZy5mYXVsdF90eXBlID09ICJOb3JtYWwiKSAmIChjYXRhbG9nLnJwbS5pc2luKGNvbmZpZy5kYXRhLnRyYWluX3JwbXMpKQogICAgXQogICAgaWYgbGVuKHRyYWluX3Jvd3MpICE9IGxlbihjb25maWcuZGF0YS50cmFpbl9ycG1zKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgb25lIE5vcm1hbC5ucHogcGVyIHRyYWluIFJQTSB7Y29uZmlnLmRhdGEudHJhaW5fcnBtc30sICIKICAgICAgICAgICAgZiJmb3VuZCB7bGVuKHRyYWluX3Jvd3MpfSBtYXRjaGluZyBmaWxlcy4iCiAgICAgICAgKQogICAgdHJhaW5fWCwgXyA9IF9zcGVjdHJvZ3JhbXNfZm9yX3Jvd3ModHJhaW5fcm93cywgY29uZmlnLCBjb25maWcud2luZG93aW5nLnRyYWluX3N0cmlkZSkKCiAgICAjIC0tLSBIT0xET1VUIGhlYWx0aHk6IE5vcm1hbCBmaWxlIGZyb20gaG9sZG91dF9ycG0sIHNwYXJzZSBvdmVybGFwLCBzcGxpdCBpbiB0d28gLS0tCiAgICBob2xkb3V0X3Jvd3MgPSBjYXRhbG9nWwogICAgICAgIChjYXRhbG9nLmZhdWx0X3R5cGUgPT0gIk5vcm1hbCIpICYgKGNhdGFsb2cucnBtID09IGNvbmZpZy5kYXRhLmhvbGRvdXRfcnBtKQogICAgXQogICAgaWYgbGVuKGhvbGRvdXRfcm93cykgIT0gMToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYiRXhwZWN0ZWQgZXhhY3RseSBvbmUgTm9ybWFsLm5weiBmb3IgaG9sZG91dCBSUE0ge2NvbmZpZy5kYXRhLmhvbGRvdXRfcnBtfSwgIgogICAgICAgICAgICBmImZvdW5kIHtsZW4oaG9sZG91dF9yb3dzKX0uIgogICAgICAgICkKICAgIGhvbGRvdXRfWCwgaG9sZG91dF9tZXRhID0gX3NwZWN0cm9ncmFtc19mb3Jfcm93cyhob2xkb3V0X3Jvd3MsIGNvbmZpZywgY29uZmlnLndpbmRvd2luZy5ldmFsX3N0cmlkZSkKICAgIG5fdmFsID0gaW50KHJvdW5kKGxlbihob2xkb3V0X1gpICogY29uZmlnLnRyYWluLnZhbF9mcmFjdGlvbl9vZl9ob2xkb3V0KSkKICAgIHZhbF9YLCB0ZXN0X2hlYWx0aHlfWCA9IGhvbGRvdXRfWFs6bl92YWxdLCBob2xkb3V0X1hbbl92YWw6XQogICAgdmFsX21ldGEsIHRlc3RfaGVhbHRoeV9tZXRhID0gaG9sZG91dF9tZXRhLmlsb2NbOm5fdmFsXS5jb3B5KCksIGhvbGRvdXRfbWV0YS5pbG9jW25fdmFsOl0uY29weSgpCgogICAgIyAtLS0gVEVTVCBmYXVsdHk6IGV2ZXJ5IGZhdWx0IGZpbGUgZnJvbSBmYXVsdF9ycG1zIHggZmF1bHRfdHlwZXMgLS0tCiAgICBmYXVsdF9yb3dzID0gY2F0YWxvZ1sKICAgICAgICBjYXRhbG9nLmZhdWx0X3R5cGUuaXNpbihjb25maWcuZGF0YS5mYXVsdF90eXBlcykgJiBjYXRhbG9nLnJwbS5pc2luKGNvbmZpZy5kYXRhLmZhdWx0X3JwbXMpCiAgICBdCiAgICBpZiBsZW4oZmF1bHRfcm93cykgPT0gMDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5vIGZhdWx0eSBmaWxlcyBtYXRjaGVkIGNvbmZpZy5kYXRhLmZhdWx0X3R5cGVzL2ZhdWx0X3JwbXMuIikKICAgIGZhdWx0X1gsIGZhdWx0X21ldGEgPSBfc3BlY3Ryb2dyYW1zX2Zvcl9yb3dzKGZhdWx0X3Jvd3MsIGNvbmZpZywgY29uZmlnLndpbmRvd2luZy5ldmFsX3N0cmlkZSkKCiAgICB0ZXN0X1ggPSBucC5jb25jYXRlbmF0ZShbdGVzdF9oZWFsdGh5X1gsIGZhdWx0X1hdLCBheGlzPTApCiAgICB0ZXN0X21ldGEgPSBwZC5jb25jYXQoW3Rlc3RfaGVhbHRoeV9tZXRhLCBmYXVsdF9tZXRhXSwgYXhpcz0wLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKCiAgICBucC5zYXZlKHBhdGhzWyJ0cmFpbl9YIl0sIHRyYWluX1gpCiAgICBucC5zYXZlKHBhdGhzWyJ2YWxfWCJdLCB2YWxfWCkKICAgIHZhbF9tZXRhLnRvX2NzdihwYXRoc1sidmFsX21ldGEiXSwgaW5kZXg9RmFsc2UpCiAgICBucC5zYXZlKHBhdGhzWyJ0ZXN0X1giXSwgdGVzdF9YKQogICAgdGVzdF9tZXRhLnRvX2NzdihwYXRoc1sidGVzdF9tZXRhIl0sIGluZGV4PUZhbHNlKQoKICAgIHJldHVybiB0cmFpbl9YLCB2YWxfWCwgdmFsX21ldGEsIHRlc3RfWCwgdGVzdF9tZXRhCg=='
    ),
    'src/vibration_ad/data/download.py': (
        'IiIiRG93bmxvYWQgdGhlIENXUlUgYmVhcmluZyAubnB6IGZpbGVzIHdlIG5lZWQgZnJvbSB0aGUgc3JpZ2FzL0NXUlVfQmVhcmluZ19OdW1QeQpHaXRIdWIgbWlycm9yLgoKRGVsaWJlcmF0ZWx5IHN0ZGxpYi1vbmx5ICh1cmxsaWIpLCBubyBgcmVxdWVzdHNgIGRlcGVuZGVuY3k6IHRoaXMgaXMgdGhlIG9uZQpwYXJ0IG9mIHRoZSBwaXBlbGluZSBzb21lb25lIG1pZ2h0IHdhbnQgdG8gcnVuIGJlZm9yZSBzZXR0aW5nIHVwIHRoZSBmdWxsCnZlbnYsIHNvIGl0IHNob3VsZG4ndCBuZWVkIGFueXRoaW5nIGJleW9uZCBhIGJhc2UgUHl0aG9uIDMgaW50ZXJwcmV0ZXIuCgpUaGUgZXhhY3Qgc2V0IG9mIGZpbGVzIGlzIGxpc3RlZCBpbiBkYXRhL3Jhdy9NQU5JRkVTVC50eHQgKG9uZSByZXBvLXJlbGF0aXZlCnBhdGggcGVyIGxpbmUpIHJhdGhlciB0aGFuIGRpc2NvdmVyZWQgZHluYW1pY2FsbHksIHNvIGEgcnVuIGlzIHJlcHJvZHVjaWJsZQpldmVuIGlmIHRoZSB1cHN0cmVhbSByZXBvIGNoYW5nZXMgbGF0ZXIuIE9ubHkgdGhlIDEyIGtIeiBEcml2ZS1FbmQgZmF1bHQKZmlsZXMgKCpfREUxMi5ucHopIGFuZCB0aGUgTm9ybWFsIGJhc2VsaW5lIGZpbGVzIGFyZSBsaXN0ZWQgLS0gc2VlCm5vdGVzLzAxX2RhdGFzZXRfbm90ZXMubWQgZm9yIHdoeS4KClVzYWdlOgogICAgcHl0aG9uIC1tIHZpYnJhdGlvbl9hZC5kYXRhLmRvd25sb2FkIC0tY29uZmlnIGNvbmZpZ3MvZGVmYXVsdC55YW1sCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHNodXRpbAppbXBvcnQgc3lzCmltcG9ydCB1cmxsaWIuZXJyb3IKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSB2aWJyYXRpb25fYWQuY29uZmlnIGltcG9ydCBsb2FkX2NvbmZpZwoKUkFXX0JBU0VfVVJMID0gImh0dHBzOi8vcmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbS9zcmlnYXMvQ1dSVV9CZWFyaW5nX051bVB5L21haW4vIgpSRVFVRVNUX1RJTUVPVVRfUyA9IDE1CiMgSWYgdGhpcyBtYW55IGRvd25sb2FkcyBpbiBhIHJvdyBmYWlsLCBzdG9wIGluc3RlYWQgb2YgZ3JpbmRpbmcgdGhyb3VnaCB0aGUKIyByZXN0IG9mIHRoZSBtYW5pZmVzdCBhdCB0aGUgc2FtZSAodXN1YWxseSB+MzBzIE9TIHJlc29sdmVyIHRpbWVvdXQpIGNvc3QKIyBwZXIgZmlsZSAtLSBhIHN5c3RlbWljIHByb2JsZW0gKG5vIGludGVybmV0LCBETlMgYnJva2VuLCBHaXRIdWIgYmxvY2tlZCkKIyBpc24ndCBnb2luZyB0byBzdGFydCBzdWNjZWVkaW5nIG9uIGZpbGUgTisxIGp1c3QgYmVjYXVzZSBpdCBmYWlsZWQgb24gTi4KTUFYX0NPTlNFQ1VUSVZFX0ZBSUxVUkVTID0gMwoKCmRlZiBkb3dubG9hZF9tYW5pZmVzdChtYW5pZmVzdF9wYXRoOiBQYXRoLCBkZXN0X2RpcjogUGF0aCkgLT4gbGlzdFtzdHJdOgogICAgIiIiUmV0dXJucyB0aGUgbGlzdCBvZiByZXBvLXJlbGF0aXZlIHBhdGhzIHRoYXQgZmFpbGVkIHRvIGRvd25sb2FkLiIiIgogICAgZGVzdF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgbGluZXMgPSBbCiAgICAgICAgbGluZS5zdHJpcCgpCiAgICAgICAgZm9yIGxpbmUgaW4gbWFuaWZlc3RfcGF0aC5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkKICAgICAgICBpZiBsaW5lLnN0cmlwKCkKICAgIF0KCiAgICBmYWlsZWQ6IGxpc3Rbc3RyXSA9IFtdCiAgICBjb25zZWN1dGl2ZV9mYWlsdXJlcyA9IDAKICAgIGZvciBpLCByZXBvX3JlbHBhdGggaW4gZW51bWVyYXRlKGxpbmVzLCAxKToKICAgICAgICBmbmFtZSA9IHJlcG9fcmVscGF0aC5yc3BsaXQoIi8iLCAxKVstMV0KICAgICAgICBkZXN0ID0gZGVzdF9kaXIgLyBmbmFtZQogICAgICAgIGlmIGRlc3QuZXhpc3RzKCk6CiAgICAgICAgICAgIHByaW50KGYiW3tpfS97bGVuKGxpbmVzKX1dIHNraXAgKGV4aXN0cyk6IHtmbmFtZX0iKQogICAgICAgICAgICBjb25zZWN1dGl2ZV9mYWlsdXJlcyA9IDAKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgdXJsID0gUkFXX0JBU0VfVVJMICsgdXJsbGliLnJlcXVlc3QucXVvdGUocmVwb19yZWxwYXRoKQogICAgICAgIHByaW50KGYiW3tpfS97bGVuKGxpbmVzKX1dIGRvd25sb2FkaW5nOiB7Zm5hbWV9IikKICAgICAgICB0bXAgPSBkZXN0LndpdGhfc3VmZml4KGRlc3Quc3VmZml4ICsgIi5wYXJ0IikKICAgICAgICB0cnk6CiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3Blbih1cmwsIHRpbWVvdXQ9UkVRVUVTVF9USU1FT1VUX1MpIGFzIHJlc3BvbnNlLCBvcGVuKHRtcCwgIndiIikgYXMgZjoKICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZW9iaihyZXNwb25zZSwgZikKICAgICAgICAgICAgdG1wLnJlbmFtZShkZXN0KQogICAgICAgICAgICBjb25zZWN1dGl2ZV9mYWlsdXJlcyA9IDAKICAgICAgICBleGNlcHQgKHVybGxpYi5lcnJvci5VUkxFcnJvciwgVGltZW91dEVycm9yKSBhcyBlOgogICAgICAgICAgICBwcmludChmIiAgRkFJTEVEOiB7cmVwb19yZWxwYXRofSAoe2V9KSIsIGZpbGU9c3lzLnN0ZGVycikKICAgICAgICAgICAgdG1wLnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgICAgIGZhaWxlZC5hcHBlbmQocmVwb19yZWxwYXRoKQogICAgICAgICAgICBjb25zZWN1dGl2ZV9mYWlsdXJlcyArPSAxCiAgICAgICAgICAgIGlmIGNvbnNlY3V0aXZlX2ZhaWx1cmVzID49IE1BWF9DT05TRUNVVElWRV9GQUlMVVJFUzoKICAgICAgICAgICAgICAgIHByaW50KAogICAgICAgICAgICAgICAgICAgIGYiQWJvcnRpbmcgYWZ0ZXIge2NvbnNlY3V0aXZlX2ZhaWx1cmVzfSBjb25zZWN1dGl2ZSBmYWlsdXJlcyAiCiAgICAgICAgICAgICAgICAgICAgIi0tIGxpa2VseSBubyBpbnRlcm5ldCBhY2Nlc3MgcmF0aGVyIHRoYW4gYSBwZXItZmlsZSBpc3N1ZSAiCiAgICAgICAgICAgICAgICAgICAgIihvbiBLYWdnbGU6IGNoZWNrIHRoZSBub3RlYm9vaydzIEludGVybmV0IHRvZ2dsZSBBTkQgdGhhdCAiCiAgICAgICAgICAgICAgICAgICAgInRoZSBhY2NvdW50IGlzIHBob25lLXZlcmlmaWVkLCB3aGljaCBnYXRlcyBpbnRlcm5ldC9HUFUgIgogICAgICAgICAgICAgICAgICAgICJhY2Nlc3MgcmVnYXJkbGVzcyBvZiB0aGF0IHRvZ2dsZSkuIiwKICAgICAgICAgICAgICAgICAgICBmaWxlPXN5cy5zdGRlcnIsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBmYWlsZWQuZXh0ZW5kKGxpbmVzW2k6XSkgICMgcmVtYWluaW5nLCB1bi1hdHRlbXB0ZWQgZmlsZXMgYWxzbyAiZmFpbGVkIgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICByZXR1cm4gZmFpbGVkCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlncy9kZWZhdWx0LnlhbWwiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKCiAgICBjb25maWcgPSBsb2FkX2NvbmZpZyhhcmdzLmNvbmZpZykKICAgIHJhd19kaXIgPSBjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMucmF3X2RpcikKICAgIG1hbmlmZXN0X3BhdGggPSByYXdfZGlyIC8gIk1BTklGRVNULnR4dCIKCiAgICBpZiBub3QgbWFuaWZlc3RfcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJNYW5pZmVzdCBub3QgZm91bmQgYXQge21hbmlmZXN0X3BhdGh9LiBJdCBzaG91bGQgYmUgY2hlY2tlZCAiCiAgICAgICAgICAgICJpbnRvIHRoZSByZXBvIGFsb25nc2lkZSB0aGlzIHNjcmlwdC4iCiAgICAgICAgKQoKICAgIGZhaWxlZCA9IGRvd25sb2FkX21hbmlmZXN0KG1hbmlmZXN0X3BhdGgsIHJhd19kaXIpCiAgICBuX2ZpbGVzID0gbGVuKGxpc3QocmF3X2Rpci5nbG9iKCIqLm5weiIpKSkKICAgIHByaW50KGYiRG9uZS4ge25fZmlsZXN9IC5ucHogZmlsZXMgcHJlc2VudCBpbiB7cmF3X2Rpcn0iKQoKICAgIGlmIGZhaWxlZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgIGYie2xlbihmYWlsZWQpfSBmaWxlKHMpIGZhaWxlZCB0byBkb3dubG9hZCwgZS5nLiB7ZmFpbGVkWzBdIXJ9LiAiCiAgICAgICAgICAgICJOb3QgY29udGludWluZyB3aXRoIGEgcGFydGlhbC9lbXB0eSBkYXRhc2V0IC0tIHNlZSBzdGRlcnIgYWJvdmUgIgogICAgICAgICAgICAiZm9yIHRoZSB1bmRlcmx5aW5nIGVycm9yLiIKICAgICAgICApCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'src/vibration_ad/data/torch_dataset.py': (
        'IiIiVGhpbiB0b3JjaC51dGlscy5kYXRhLkRhdGFzZXQgd3JhcHBlciBhcm91bmQgYSBub3JtYWxpemVkIHNwZWN0cm9ncmFtIGFycmF5LgoKS2VwdCBzZXBhcmF0ZSBmcm9tIGRhdGFzZXQucHkgc28gdGhlIGFjdHVhbCBkYXRhLWVuZ2luZWVyaW5nIGNvZGUgKHdpbmRvd2luZywKU1RGVCwgc3BsaXR0aW5nLCBjYWNoaW5nKSBoYXMgbm8gdG9yY2ggZGVwZW5kZW5jeSBhbmQgY2FuIGJlIHVuaXQtdGVzdGVkIG9yCnJldXNlZCB3aXRob3V0IGEgZGVlcCBsZWFybmluZyBmcmFtZXdvcmsgaW5zdGFsbGVkLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCgpjbGFzcyBTcGVjdHJvZ3JhbURhdGFzZXQoRGF0YXNldCk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgc3BlY3Ryb2dyYW1zOiBucC5uZGFycmF5KToKICAgICAgICAjIChOLCBGLCBUKSAtPiAoTiwgMSwgRiwgVCk6IGFkZCB0aGUgc2luZ2xlLWNoYW5uZWwgZGltIGNvbnYyZCBleHBlY3RzLgogICAgICAgIHNlbGYuZGF0YSA9IHRvcmNoLmZyb21fbnVtcHkoc3BlY3Ryb2dyYW1zKS51bnNxdWVlemUoMSkuZmxvYXQoKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZGF0YSkKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4OiBpbnQpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gc2VsZi5kYXRhW2lkeF0K'
    ),
    'src/vibration_ad/data/windows.py': (
        'IiIiQ3V0IGEgbG9uZyAxRCB2aWJyYXRpb24gc2lnbmFsIGludG8gZml4ZWQtbGVuZ3RoIG92ZXJsYXBwaW5nIHdpbmRvd3MuCgpUaGlzIGlzIHRoZSBzdGVwIHRoYXQgdHVybnMgIm9uZSBsb25nIHJlY29yZGluZyBwZXIgYmVhcmluZyBjb25kaXRpb24iCmludG8gIm1hbnkgaW5kZXBlbmRlbnQgZXhhbXBsZXMiIGZvciB0aGUgYXV0b2VuY29kZXIgdG8gdHJhaW4vZXZhbHVhdGUgb24uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgbWFrZV93aW5kb3dzKHNpZ25hbDogbnAubmRhcnJheSwgc2VnbWVudF9sZW5ndGg6IGludCwgc3RyaWRlOiBpbnQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJTbGljZSBgc2lnbmFsYCBpbnRvIG92ZXJsYXBwaW5nIHdpbmRvd3Mgb2YgbGVuZ3RoIGBzZWdtZW50X2xlbmd0aGAuCgogICAgUmV0dXJucyBhbiBhcnJheSBvZiBzaGFwZSAobl93aW5kb3dzLCBzZWdtZW50X2xlbmd0aCkuIFVzZXMgYSBzdHJpZGVkCiAgICB2aWV3IChubyBjb3B5KSB2aWEgbnVtcHkncyBzbGlkaW5nX3dpbmRvd192aWV3IGZvciBzcGVlZCwgdGhlbiB0YWtlcwogICAgZXZlcnkgYHN0cmlkZWAtdGggd2luZG93LgoKICAgIEFueSBzYW1wbGVzIHRoYXQgZG9uJ3QgZmlsbCBhIGZpbmFsIGNvbXBsZXRlIHdpbmRvdyBhcmUgZHJvcHBlZC4KICAgICIiIgogICAgaWYgbGVuKHNpZ25hbCkgPCBzZWdtZW50X2xlbmd0aDoKICAgICAgICByZXR1cm4gbnAuZW1wdHkoKDAsIHNlZ21lbnRfbGVuZ3RoKSwgZHR5cGU9c2lnbmFsLmR0eXBlKQoKICAgIGFsbF93aW5kb3dzID0gbnAubGliLnN0cmlkZV90cmlja3Muc2xpZGluZ193aW5kb3dfdmlldyhzaWduYWwsIHNlZ21lbnRfbGVuZ3RoKQogICAgcmV0dXJuIG5wLmFzY29udGlndW91c2FycmF5KGFsbF93aW5kb3dzWzo6c3RyaWRlXSkKCgpkZWYgbl93aW5kb3dzKHNpZ25hbF9sZW5ndGg6IGludCwgc2VnbWVudF9sZW5ndGg6IGludCwgc3RyaWRlOiBpbnQpIC0+IGludDoKICAgICIiIk51bWJlciBvZiB3aW5kb3dzIG1ha2Vfd2luZG93cygpIHdvdWxkIHByb2R1Y2UsIHdpdGhvdXQgYnVpbGRpbmcgdGhlbS4iIiIKICAgIGlmIHNpZ25hbF9sZW5ndGggPCBzZWdtZW50X2xlbmd0aDoKICAgICAgICByZXR1cm4gMAogICAgcmV0dXJuIChzaWduYWxfbGVuZ3RoIC0gc2VnbWVudF9sZW5ndGgpIC8vIHN0cmlkZSArIDEK'
    ),
    'src/vibration_ad/features/__init__.py': (
        ''
    ),
    'src/vibration_ad/features/spectrogram.py': (
        'IiIiVHVybiByYXcgc2lnbmFsIHdpbmRvd3MgaW50byBub3JtYWxpemVkIGxvZy1tYWduaXR1ZGUgU1RGVCBzcGVjdHJvZ3JhbXMuCgpXZSB1c2UgYSBwbGFpbiBTVEZUIChsaW5lYXIgZnJlcXVlbmN5IGF4aXMpIHJhdGhlciB0aGFuIGEgTWVsLXNwZWN0cm9ncmFtLgpNZWwgc2NhbGluZyB3YXJwcyB0aGUgZnJlcXVlbmN5IGF4aXMgdG8gbWF0Y2ggaHVtYW4gcGl0Y2ggcGVyY2VwdGlvbiwgd2hpY2gKaXMgdGhlIHJpZ2h0IHRoaW5nIGZvciBzcGVlY2gvbXVzaWMgYnV0IGFjdGl2ZWx5IHVuaGVscGZ1bCBoZXJlOiBiZWFyaW5nCmZhdWx0IGVuZXJneSBzaG93cyB1cCBhdCBzcGVjaWZpYyBtZWNoYW5pY2FsIGZyZXF1ZW5jaWVzIChCUEZPLCBCUEZJLCBCU0YsCnNoYWZ0IHJhdGUgYW5kIHRoZWlyIGhhcm1vbmljcyAtLSBzZWUgbm90ZXMvMDVfZ2xvc3NhcnkubWQpLCBhbmQgTWVsIGJpbm5pbmcKd291bGQgYmx1ciBleGFjdGx5IHRoZSBoaWdoZXItZnJlcXVlbmN5IGRldGFpbCB0aGF0IHNlcGFyYXRlcyBvbmUgZmF1bHQKdHlwZS9zZXZlcml0eSBmcm9tIGFub3RoZXIuIExpbmVhciBTVEZUIGtlZXBzIHRoYXQgcmVzb2x1dGlvbiBpbnRhY3QgYW5kCmFsc28gZHJvcHMgdGhlIGxpYnJvc2EgZGVwZW5kZW5jeSBlbnRpcmVseSAoc2NpcHkuc2lnbmFsLnN0ZnQgaXMgZW5vdWdoKS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNjaXB5LnNpZ25hbCBpbXBvcnQgc3RmdAoKZnJvbSB2aWJyYXRpb25fYWQuY29uZmlnIGltcG9ydCBTVEZUQ29uZmlnLCBXaW5kb3dpbmdDb25maWcKCgpkZWYgZXhwZWN0ZWRfc2hhcGUod2luZG93aW5nOiBXaW5kb3dpbmdDb25maWcsIHN0ZnRfY2ZnOiBTVEZUQ29uZmlnKSAtPiB0dXBsZVtpbnQsIGludF06CiAgICAiIiIobl9mcmVxX2JpbnMsIG5fdGltZV9mcmFtZXMpIHRoYXQgd2luZG93c190b19sb2dfc3BlY3Ryb2dyYW1zKCkgd2lsbAogICAgcHJvZHVjZSBmb3IgdGhlIGdpdmVuIGNvbmZpZywgY29tcHV0ZWQgd2l0aG91dCBydW5uaW5nIGFueSBTVEZULgoKICAgIEZyZXEgYmlucyA9IG5wZXJzZWcgLy8gMiAodGhlIE55cXVpc3QgYmluIGlzIGRyb3BwZWQsIHNlZSBiZWxvdykuCiAgICBUaW1lIGZyYW1lcyBmb2xsb3dzIHNjaXB5J3Mgbm8tcGFkZGluZywgbm8tYm91bmRhcnkgU1RGVCBmcmFtZSBjb3VudDoKICAgIDEgKyAoc2VnbWVudF9sZW5ndGggLSBucGVyc2VnKSAvLyBob3BfbGVuZ3RoLgogICAgIiIiCiAgICBuX2ZyZXEgPSBzdGZ0X2NmZy5ucGVyc2VnIC8vIDIKICAgIG5fdGltZSA9ICh3aW5kb3dpbmcuc2VnbWVudF9sZW5ndGggLSBzdGZ0X2NmZy5ucGVyc2VnKSAvLyBzdGZ0X2NmZy5ob3BfbGVuZ3RoICsgMQogICAgcmV0dXJuIG5fZnJlcSwgbl90aW1lCgoKZGVmIHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcygKICAgIHdpbmRvd3M6IG5wLm5kYXJyYXksIGZzX2h6OiBpbnQsIHN0ZnRfY2ZnOiBTVEZUQ29uZmlnCikgLT4gbnAubmRhcnJheToKICAgICIiIndpbmRvd3M6IChOLCBzZWdtZW50X2xlbmd0aCkgLT4gbG9nLW1hZ25pdHVkZSBzcGVjdHJvZ3JhbXMgKE4sIEYsIFQpLgoKICAgIGJvdW5kYXJ5PU5vbmUsIHBhZGRlZD1GYWxzZSBzbyB0aGUgb3V0cHV0IHRpbWUtZnJhbWUgY291bnQgaXMgZXhhY3RseQogICAgcHJlZGljdGFibGUgZnJvbSAoc2VnbWVudF9sZW5ndGgsIG5wZXJzZWcsIGhvcF9sZW5ndGgpIC0tIHNlZQogICAgZXhwZWN0ZWRfc2hhcGUoKS4gVGhlIE55cXVpc3QgYmluIChsYXN0IGZyZXEgYmluKSBpcyBkcm9wcGVkIG9ubHkgdG8KICAgIG1ha2UgRiBhIHJvdW5kLCBwb29saW5nLWZyaWVuZGx5IG51bWJlciBmb3IgdGhlIGNvbnYgYXV0b2VuY29kZXI7IGZvcgogICAgdGhlc2Ugc2lnbmFscyBpdCBjYXJyaWVzIG5lZ2xpZ2libGUgZW5lcmd5IGFueXdheS4KICAgICIiIgogICAgXywgXywgWnh4ID0gc3RmdCgKICAgICAgICB3aW5kb3dzLAogICAgICAgIGZzPWZzX2h6LAogICAgICAgIHdpbmRvdz0iaGFubiIsCiAgICAgICAgbnBlcnNlZz1zdGZ0X2NmZy5ucGVyc2VnLAogICAgICAgIG5vdmVybGFwPXN0ZnRfY2ZnLm5vdmVybGFwLAogICAgICAgIGJvdW5kYXJ5PU5vbmUsCiAgICAgICAgcGFkZGVkPUZhbHNlLAogICAgICAgIGF4aXM9LTEsCiAgICApCiAgICBtYWduaXR1ZGUgPSBucC5hYnMoWnh4KVs6LCA6LTEsIDpdICAjIGRyb3AgTnlxdWlzdCBiaW4gLT4gKE4sIG5wZXJzZWcvMiwgVCkKICAgIHJldHVybiBucC5sb2cobWFnbml0dWRlICsgc3RmdF9jZmcubG9nX2Vwc2lsb24pLmFzdHlwZShucC5mbG9hdDMyKQoKCkBkYXRhY2xhc3MKY2xhc3MgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyOgogICAgIiIiUGVyLWZyZXF1ZW5jeS1iaW4gei1zY29yZSBub3JtYWxpemF0aW9uLCBmaXQgb24gVFJBSU4gaGVhbHRoeSBkYXRhIG9ubHkuCgogICAgUGVyLWJpbiAobm90IG9uZSBnbG9iYWwgc2NhbGFyKSBtYXR0ZXJzIGhlcmU6IGxvdyBmcmVxdWVuY2llcyBjYXJyeSBmYXIKICAgIG1vcmUgZW5lcmd5IHRoYW4gaGlnaCBmcmVxdWVuY2llcyBpbiB0aGVzZSBzaWduYWxzLCBzbyBhIHNpbmdsZSBnbG9iYWwKICAgIG1lYW4vc3RkIHdvdWxkIGxlYXZlIGhpZ2gtZnJlcXVlbmN5IGJpbnMgLS0gd2hlcmUgc3VidGxlIGZhdWx0IGhhcm1vbmljcwogICAgb2Z0ZW4gc2hvdyB1cCAtLSBhbG1vc3QgZmxhdCBhZnRlciBub3JtYWxpemF0aW9uLiBGaXR0aW5nIG9ubHkgb24KICAgIHRyYWluaW5nLXNwbGl0IGhlYWx0aHkgc3BlY3Ryb2dyYW1zIChuZXZlciB2YWwvdGVzdC9mYXVsdHkgZGF0YSkgYXZvaWRzCiAgICBsZWFraW5nIGFueSBpbmZvcm1hdGlvbiBhYm91dCB0aGUgZXZhbHVhdGlvbiBzZXQgaW50byBwcmVwcm9jZXNzaW5nLgogICAgIiIiCgogICAgbWVhbjogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lICAjIHNoYXBlIChGLCAxKQogICAgc3RkOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUgICMgc2hhcGUgKEYsIDEpCgogICAgZGVmIGZpdChzZWxmLCBzcGVjdHJvZ3JhbXM6IG5wLm5kYXJyYXkpIC0+ICJTcGVjdHJvZ3JhbU5vcm1hbGl6ZXIiOgogICAgICAgIHNlbGYubWVhbiA9IHNwZWN0cm9ncmFtcy5tZWFuKGF4aXM9KDAsIDIpLCBrZWVwZGltcz1UcnVlKVswXQogICAgICAgIHNlbGYuc3RkID0gc3BlY3Ryb2dyYW1zLnN0ZChheGlzPSgwLCAyKSwga2VlcGRpbXM9VHJ1ZSlbMF0gKyAxZS04CiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYgdHJhbnNmb3JtKHNlbGYsIHNwZWN0cm9ncmFtczogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICBpZiBzZWxmLm1lYW4gaXMgTm9uZSBvciBzZWxmLnN0ZCBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIlNwZWN0cm9ncmFtTm9ybWFsaXplci5maXQoKSBtdXN0IGJlIGNhbGxlZCBiZWZvcmUgdHJhbnNmb3JtKCkuIikKICAgICAgICByZXR1cm4gKChzcGVjdHJvZ3JhbXMgLSBzZWxmLm1lYW4pIC8gc2VsZi5zdGQpLmFzdHlwZShucC5mbG9hdDMyKQoKICAgIGRlZiBzYXZlKHNlbGYsIHBhdGg6IFBhdGgpIC0+IE5vbmU6CiAgICAgICAgbnAuc2F2ZXoocGF0aCwgbWVhbj1zZWxmLm1lYW4sIHN0ZD1zZWxmLnN0ZCkKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBsb2FkKGNscywgcGF0aDogUGF0aCkgLT4gIlNwZWN0cm9ncmFtTm9ybWFsaXplciI6CiAgICAgICAgZGF0YSA9IG5wLmxvYWQocGF0aCkKICAgICAgICByZXR1cm4gY2xzKG1lYW49ZGF0YVsibWVhbiJdLCBzdGQ9ZGF0YVsic3RkIl0pCg=='
    ),
    'src/vibration_ad/models/__init__.py': (
        ''
    ),
    'src/vibration_ad/models/autoencoder.py': (
        'IiIiQ29udm9sdXRpb25hbCBhdXRvZW5jb2RlcjogdGhlICJicmFpbiIgb2YgdGhlIGRldGVjdG9yLgoKVHJhaW5lZCBPTkxZIG9uIHNwZWN0cm9ncmFtcyBvZiBoZWFsdGh5IHZpYnJhdGlvbiBzaWduYWwgKHNlZQpkYXRhL2RhdGFzZXQucHkncyB0cmFpbiBzcGxpdCkuIFRoZSBpZGVhOiBpZiB0aGUgbmV0d29yayBvbmx5IGV2ZXIgc2VlcyBhCmhlYWx0aHkgbW90b3IncyBzcGVjdHJhbCBzaWduYXR1cmUsIGl0cyBib3R0bGVuZWNrIGxlYXJucyBhIGNvbXBhY3QgY29kZQpmb3IgIndoYXQgaGVhbHRoeSBsb29rcyBsaWtlIiAtLSBhbmQgcmVjb25zdHJ1Y3Rpb24gcXVhbGl0eSBkZWdyYWRlcyBvbgphbnl0aGluZyB0aGF0IGRvZXNuJ3QgZml0IHRoYXQgcGF0dGVybiwgaS5lLiBhIGZhdWx0LiBUaGUgcmVjb25zdHJ1Y3Rpb24KZXJyb3IgKHNlZSBwZXJfc2FtcGxlX21zZSBiZWxvdykgaXMgdGhlIGFub21hbHkgc2NvcmU7IHNlZSBldmFsdWF0ZS5weSBmb3IKaG93IHRoZSBkZXRlY3Rpb24gdGhyZXNob2xkIGlzIGNob3NlbiBhbmQgbW9kZWxzX25vdGVzIGluCm5vdGVzLzAzX21vZGVsX25vdGVzLm1kIGZvciB0aGUgYm90dGxlbmVjay1zaXplIHRyYWRlb2ZmLgoKQXJjaGl0ZWN0dXJlOiA0IHN0cmlkZS0yIGNvbnYgbGF5ZXJzIGRvd24gdG8gYW4gOHg4IGZlYXR1cmUgbWFwLCBhIGRlbnNlCmJvdHRsZW5lY2ssIHRoZW4gdGhlIG1pcnJvciBpbWFnZSBpbiBDb252VHJhbnNwb3NlMmQgbGF5ZXJzIGJhY2sgdXAuIFRoaXMKYXNzdW1lcyBhIHNxdWFyZSwgcG93ZXItb2YtdHdvIGlucHV0ICgxMjh4MTI4IGJ5IGRlZmF1bHQgLS0gc2VlCmNvbmZpZ3MvZGVmYXVsdC55YW1sIGFuZCBmZWF0dXJlcy9zcGVjdHJvZ3JhbS5leHBlY3RlZF9zaGFwZSkgc28gZWFjaApjb252L2RlY29udiBsYXllciBleGFjdGx5IGhhbHZlcy9kb3VibGVzIHNwYXRpYWwgc2l6ZSB3aXRoIG5vIHJvdW5kaW5nLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCmNsYXNzIENvbnZBdXRvZW5jb2Rlcihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGlucHV0X3NoYXBlOiB0dXBsZVtpbnQsIGludF0sIGJhc2VfY2hhbm5lbHM6IGludCA9IDE2LCBsYXRlbnRfZGltOiBpbnQgPSAxMjgpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGZyZXFfYmlucywgdGltZV9mcmFtZXMgPSBpbnB1dF9zaGFwZQogICAgICAgIGlmIGZyZXFfYmlucyAlIDE2ICE9IDAgb3IgdGltZV9mcmFtZXMgJSAxNiAhPSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJpbnB1dF9zaGFwZSB7aW5wdXRfc2hhcGV9IG11c3QgYmUgZGl2aXNpYmxlIGJ5IDE2IChmb3VyIHN0cmlkZS0yIGxheWVycyk7ICIKICAgICAgICAgICAgICAgICJhZGp1c3QgY29uZmlncy9kZWZhdWx0LnlhbWwncyBzdGZ0L3dpbmRvd2luZyBwYXJhbXMuIgogICAgICAgICAgICApCiAgICAgICAgYyA9IGJhc2VfY2hhbm5lbHMKCiAgICAgICAgc2VsZi5lbmNvZGVyX2NvbnYgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoMSwgYywga2VybmVsX3NpemU9Mywgc3RyaWRlPTIsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGMpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChjLCBjICogMiwga2VybmVsX3NpemU9Mywgc3RyaWRlPTIsIHBhZGRpbmc9MSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKGMgKiAyKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICBubi5Db252MmQoYyAqIDIsIGMgKiA0LCBrZXJuZWxfc2l6ZT0zLCBzdHJpZGU9MiwgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoYyAqIDQpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChjICogNCwgYyAqIDgsIGtlcm5lbF9zaXplPTMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChjICogOCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICApCiAgICAgICAgc2VsZi5yZWR1Y2VkX3NoYXBlID0gKGMgKiA4LCBmcmVxX2JpbnMgLy8gMTYsIHRpbWVfZnJhbWVzIC8vIDE2KQogICAgICAgIGZsYXRfZGltID0gYyAqIDggKiAoZnJlcV9iaW5zIC8vIDE2KSAqICh0aW1lX2ZyYW1lcyAvLyAxNikKCiAgICAgICAgc2VsZi5lbmNvZGVyX2ZjID0gbm4uTGluZWFyKGZsYXRfZGltLCBsYXRlbnRfZGltKQogICAgICAgIHNlbGYuZGVjb2Rlcl9mYyA9IG5uLkxpbmVhcihsYXRlbnRfZGltLCBmbGF0X2RpbSkKCiAgICAgICAgc2VsZi5kZWNvZGVyX2NvbnYgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252VHJhbnNwb3NlMmQoYyAqIDgsIGMgKiA0LCBrZXJuZWxfc2l6ZT00LCBzdHJpZGU9MiwgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoYyAqIDQpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnZUcmFuc3Bvc2UyZChjICogNCwgYyAqIDIsIGtlcm5lbF9zaXplPTQsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChjICogMiksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udlRyYW5zcG9zZTJkKGMgKiAyLCBjLCBrZXJuZWxfc2l6ZT00LCBzdHJpZGU9MiwgcGFkZGluZz0xKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoYyksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgIyBObyBmaW5hbCBhY3RpdmF0aW9uOiBpbnB1dHMgYXJlIHotc2NvcmVkIChyb3VnaGx5IE4oMCwxKSksIHNvCiAgICAgICAgICAgICMgdGhlIHJlY29uc3RydWN0aW9uIHRhcmdldCBpcyBhbnkgcmVhbCBudW1iZXIsIG5vdCBhIFswLDFdIHBpeGVsLgogICAgICAgICAgICBubi5Db252VHJhbnNwb3NlMmQoYywgMSwga2VybmVsX3NpemU9NCwgc3RyaWRlPTIsIHBhZGRpbmc9MSksCiAgICAgICAgKQoKICAgIGRlZiBlbmNvZGUoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgeiA9IHNlbGYuZW5jb2Rlcl9jb252KHgpCiAgICAgICAgeiA9IHouZmxhdHRlbigxKQogICAgICAgIHJldHVybiBzZWxmLmVuY29kZXJfZmMoeikKCiAgICBkZWYgZGVjb2RlKHNlbGYsIHo6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHggPSBzZWxmLmRlY29kZXJfZmMoeikKICAgICAgICB4ID0geC52aWV3KC0xLCAqc2VsZi5yZWR1Y2VkX3NoYXBlKQogICAgICAgIHJldHVybiBzZWxmLmRlY29kZXJfY29udih4KQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBzZWxmLmRlY29kZShzZWxmLmVuY29kZSh4KSkKCgpkZWYgcGVyX3NhbXBsZV9tc2UoeDogdG9yY2guVGVuc29yLCByZWNvbjogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAiIiJSZWNvbnN0cnVjdGlvbiBlcnJvciBwZXIgZXhhbXBsZTogbWVhbiBzcXVhcmVkIGVycm9yIG92ZXIgKEMsRixUKSwKICAgIGkuZS4gb25lIHNjYWxhciBhbm9tYWx5IHNjb3JlIHBlciBpbnB1dCBzcGVjdHJvZ3JhbS4gU2hhcGUgKE4sKSAtPiAoTiwpLgogICAgIiIiCiAgICByZXR1cm4gKCh4IC0gcmVjb24pICoqIDIpLm1lYW4oZGltPSgxLCAyLCAzKSkKCgpkZWYgc2VsZWN0X2RldmljZShyZXF1ZXN0ZWQ6IHN0ciA9ICJhdXRvIikgLT4gdG9yY2guZGV2aWNlOgogICAgaWYgcmVxdWVzdGVkICE9ICJhdXRvIjoKICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKHJlcXVlc3RlZCkKICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgiY3VkYSIpCiAgICBpZiB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgibXBzIikKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImNwdSIpCg=='
    ),
    'src/vibration_ad/train.py': (
        'IiIiVHJhaW4gdGhlIGF1dG9lbmNvZGVyIG9uIGhlYWx0aHktb25seSBzcGVjdHJvZ3JhbXMuCgpVc2FnZToKICAgIHB5dGhvbiAtbSB2aWJyYXRpb25fYWQudHJhaW4gLS1jb25maWcgY29uZmlncy9kZWZhdWx0LnlhbWwKClByb2R1Y2VzIChhbGwgdW5kZXIgY29uZmlnLnBhdGhzLm1vZGVsc19kaXIsIGRlZmF1bHQgb3V0cHV0cy9tb2RlbHMvKToKICAgIGF1dG9lbmNvZGVyX2Jlc3QucHQgICAtIHN0YXRlX2RpY3Qgb2YgdGhlIGJlc3QgKGxvd2VzdCB2YWwgbG9zcykgY2hlY2twb2ludAogICAgbm9ybWFsaXplci5ucHogICAgICAgICAtIHBlci1mcmVxdWVuY3ktYmluIG1lYW4vc3RkIGZpdCBvbiBUUkFJTiBzcGVjdHJvZ3JhbXMKICAgIHJ1bl9jb25maWcueWFtbCAgICAgICAgLSBjb3B5IG9mIHRoZSBjb25maWcgdXNlZCwgZm9yIHJlcHJvZHVjaWJpbGl0eQpBbmQgdW5kZXIgY29uZmlnLnBhdGhzLnJlcG9ydHNfZGlyIChkZWZhdWx0IG91dHB1dHMvcmVwb3J0cy8pOgogICAgdHJhaW5pbmdfbG9nLmNzdiAgICAgICAtIHBlci1lcG9jaCB0cmFpbi92YWwgcmVjb25zdHJ1Y3Rpb24gZXJyb3IKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcmFuZG9tCmltcG9ydCBzaHV0aWwKaW1wb3J0IHRpbWUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFMb2FkZXIKCmZyb20gdmlicmF0aW9uX2FkLmNvbmZpZyBpbXBvcnQgQ29uZmlnLCBsb2FkX2NvbmZpZwpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLmRhdGFzZXQgaW1wb3J0IGJ1aWxkX3Jhd19kYXRhc2V0cwpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLnRvcmNoX2RhdGFzZXQgaW1wb3J0IFNwZWN0cm9ncmFtRGF0YXNldApmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyLCBleHBlY3RlZF9zaGFwZQpmcm9tIHZpYnJhdGlvbl9hZC5tb2RlbHMuYXV0b2VuY29kZXIgaW1wb3J0IENvbnZBdXRvZW5jb2RlciwgcGVyX3NhbXBsZV9tc2UsIHNlbGVjdF9kZXZpY2UKCgpkZWYgc2V0X3NlZWQoc2VlZDogaW50KSAtPiBOb25lOgogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG5wLnJhbmRvbS5zZWVkKHNlZWQpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQoKCmRlZiBydW5fZXBvY2gobW9kZWwsIGxvYWRlciwgb3B0aW1pemVyLCBkZXZpY2UsIHRyYWluOiBib29sKSAtPiBmbG9hdDoKICAgIG1vZGVsLnRyYWluKHRyYWluKQogICAgdG90YWxfbG9zcywgbl9zYW1wbGVzID0gMC4wLCAwCiAgICBjb250ZXh0ID0gdG9yY2guZW5hYmxlX2dyYWQoKSBpZiB0cmFpbiBlbHNlIHRvcmNoLm5vX2dyYWQoKQogICAgd2l0aCBjb250ZXh0OgogICAgICAgIGZvciBiYXRjaCBpbiBsb2FkZXI6CiAgICAgICAgICAgIGJhdGNoID0gYmF0Y2gudG8oZGV2aWNlKQogICAgICAgICAgICByZWNvbiA9IG1vZGVsKGJhdGNoKQogICAgICAgICAgICBwZXJfc2FtcGxlID0gcGVyX3NhbXBsZV9tc2UoYmF0Y2gsIHJlY29uKQogICAgICAgICAgICBsb3NzID0gcGVyX3NhbXBsZS5tZWFuKCkKCiAgICAgICAgICAgIGlmIHRyYWluOgogICAgICAgICAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZCgpCiAgICAgICAgICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgICAgIG9wdGltaXplci5zdGVwKCkKCiAgICAgICAgICAgIHRvdGFsX2xvc3MgKz0gcGVyX3NhbXBsZS5zdW0oKS5pdGVtKCkKICAgICAgICAgICAgbl9zYW1wbGVzICs9IGxlbihiYXRjaCkKICAgIHJldHVybiB0b3RhbF9sb3NzIC8gbl9zYW1wbGVzCgoKZGVmIHRyYWluKGNvbmZpZzogQ29uZmlnLCBjb25maWdfcGF0aDogc3RyLCBmb3JjZV9kYXRhOiBib29sID0gRmFsc2UpIC0+IE5vbmU6CiAgICBzZXRfc2VlZChjb25maWcuc2VlZCkKCiAgICBtb2RlbHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLm1vZGVsc19kaXIpCiAgICByZXBvcnRzX2RpciA9IGNvbmZpZy5yZXNvbHZlKGNvbmZpZy5wYXRocy5yZXBvcnRzX2RpcikKICAgIG1vZGVsc19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmVwb3J0c19kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHByaW50KCJCdWlsZGluZyBkYXRhc2V0cyAoY2FjaGVkIGFmdGVyIGZpcnN0IHJ1bikuLi4iKQogICAgdHJhaW5fWCwgdmFsX1gsIF92YWxfbWV0YSwgX3Rlc3RfWCwgX3Rlc3RfbWV0YSA9IGJ1aWxkX3Jhd19kYXRhc2V0cyhjb25maWcsIGZvcmNlPWZvcmNlX2RhdGEpCiAgICBwcmludChmIiAgdHJhaW4gd2luZG93czoge3RyYWluX1guc2hhcGV9ICB2YWwgd2luZG93czoge3ZhbF9YLnNoYXBlfSIpCgogICAgZXhwZWN0ZWQgPSBleHBlY3RlZF9zaGFwZShjb25maWcud2luZG93aW5nLCBjb25maWcuc3RmdCkKICAgIGlmIHRyYWluX1guc2hhcGVbMTpdICE9IGV4cGVjdGVkOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlNwZWN0cm9ncmFtIHNoYXBlIHt0cmFpbl9YLnNoYXBlWzE6XX0gIT0gZXhwZWN0ZWQge2V4cGVjdGVkfSIpCgogICAgbm9ybWFsaXplciA9IFNwZWN0cm9ncmFtTm9ybWFsaXplcigpLmZpdCh0cmFpbl9YKQogICAgdHJhaW5fWF9ub3JtID0gbm9ybWFsaXplci50cmFuc2Zvcm0odHJhaW5fWCkKICAgIHZhbF9YX25vcm0gPSBub3JtYWxpemVyLnRyYW5zZm9ybSh2YWxfWCkKCiAgICBkZXZpY2UgPSBzZWxlY3RfZGV2aWNlKGNvbmZpZy50cmFpbi5kZXZpY2UpCiAgICBwcmludChmIlVzaW5nIGRldmljZToge2RldmljZX0iKQoKICAgIHRyYWluX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgU3BlY3Ryb2dyYW1EYXRhc2V0KHRyYWluX1hfbm9ybSksIGJhdGNoX3NpemU9Y29uZmlnLnRyYWluLmJhdGNoX3NpemUsIHNodWZmbGU9VHJ1ZQogICAgKQogICAgdmFsX2xvYWRlciA9IERhdGFMb2FkZXIoCiAgICAgICAgU3BlY3Ryb2dyYW1EYXRhc2V0KHZhbF9YX25vcm0pLCBiYXRjaF9zaXplPWNvbmZpZy50cmFpbi5iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlCiAgICApCgogICAgbW9kZWwgPSBDb252QXV0b2VuY29kZXIoCiAgICAgICAgaW5wdXRfc2hhcGU9ZXhwZWN0ZWQsCiAgICAgICAgYmFzZV9jaGFubmVscz1jb25maWcubW9kZWwuYmFzZV9jaGFubmVscywKICAgICAgICBsYXRlbnRfZGltPWNvbmZpZy5tb2RlbC5sYXRlbnRfZGltLAogICAgKS50byhkZXZpY2UpCiAgICBuX3BhcmFtcyA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpKQogICAgcHJpbnQoZiJNb2RlbCBwYXJhbWV0ZXJzOiB7bl9wYXJhbXM6LH0iKQoKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0oCiAgICAgICAgbW9kZWwucGFyYW1ldGVycygpLCBscj1jb25maWcudHJhaW4ubGVhcm5pbmdfcmF0ZSwgd2VpZ2h0X2RlY2F5PWNvbmZpZy50cmFpbi53ZWlnaHRfZGVjYXkKICAgICkKCiAgICBiZXN0X3ZhbF9sb3NzID0gZmxvYXQoImluZiIpCiAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA9IDAKICAgIGhpc3RvcnkgPSBbXQogICAgYmVzdF9ja3B0X3BhdGggPSBtb2RlbHNfZGlyIC8gImF1dG9lbmNvZGVyX2Jlc3QucHQiCgogICAgZm9yIGVwb2NoIGluIHJhbmdlKDEsIGNvbmZpZy50cmFpbi5lcG9jaHMgKyAxKToKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgdHJhaW5fbG9zcyA9IHJ1bl9lcG9jaChtb2RlbCwgdHJhaW5fbG9hZGVyLCBvcHRpbWl6ZXIsIGRldmljZSwgdHJhaW49VHJ1ZSkKICAgICAgICB2YWxfbG9zcyA9IHJ1bl9lcG9jaChtb2RlbCwgdmFsX2xvYWRlciwgb3B0aW1pemVyLCBkZXZpY2UsIHRyYWluPUZhbHNlKQogICAgICAgIGR0ID0gdGltZS50aW1lKCkgLSB0MAoKICAgICAgICBpbXByb3ZlZCA9IHZhbF9sb3NzIDwgYmVzdF92YWxfbG9zcwogICAgICAgIGlmIGltcHJvdmVkOgogICAgICAgICAgICBiZXN0X3ZhbF9sb3NzID0gdmFsX2xvc3MKICAgICAgICAgICAgZXBvY2hzX3dpdGhvdXRfaW1wcm92ZW1lbnQgPSAwCiAgICAgICAgICAgIHRvcmNoLnNhdmUobW9kZWwuc3RhdGVfZGljdCgpLCBiZXN0X2NrcHRfcGF0aCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCArPSAxCgogICAgICAgIG1hcmtlciA9ICIgKiIgaWYgaW1wcm92ZWQgZWxzZSAiIgogICAgICAgIHByaW50KAogICAgICAgICAgICBmImVwb2NoIHtlcG9jaDozZH0ve2NvbmZpZy50cmFpbi5lcG9jaHN9ICAiCiAgICAgICAgICAgIGYidHJhaW5fbXNlPXt0cmFpbl9sb3NzOi41Zn0gIHZhbF9tc2U9e3ZhbF9sb3NzOi41Zn0gICh7ZHQ6LjFmfXMpe21hcmtlcn0iCiAgICAgICAgKQogICAgICAgIGhpc3RvcnkuYXBwZW5kKHsiZXBvY2giOiBlcG9jaCwgInRyYWluX21zZSI6IHRyYWluX2xvc3MsICJ2YWxfbXNlIjogdmFsX2xvc3N9KQoKICAgICAgICBpZiBlcG9jaHNfd2l0aG91dF9pbXByb3ZlbWVudCA+PSBjb25maWcudHJhaW4uZWFybHlfc3RvcHBpbmdfcGF0aWVuY2U6CiAgICAgICAgICAgIHByaW50KGYiRWFybHkgc3RvcHBpbmc6IG5vIHZhbCBpbXByb3ZlbWVudCBpbiB7Y29uZmlnLnRyYWluLmVhcmx5X3N0b3BwaW5nX3BhdGllbmNlfSBlcG9jaHMuIikKICAgICAgICAgICAgYnJlYWsKCiAgICBwZC5EYXRhRnJhbWUoaGlzdG9yeSkudG9fY3N2KHJlcG9ydHNfZGlyIC8gInRyYWluaW5nX2xvZy5jc3YiLCBpbmRleD1GYWxzZSkKICAgIG5vcm1hbGl6ZXIuc2F2ZShtb2RlbHNfZGlyIC8gIm5vcm1hbGl6ZXIubnB6IikKICAgIHNodXRpbC5jb3B5KGNvbmZpZ19wYXRoLCBtb2RlbHNfZGlyIC8gInJ1bl9jb25maWcueWFtbCIpCgogICAgcHJpbnQoZiJcbkJlc3QgdmFsIE1TRToge2Jlc3RfdmFsX2xvc3M6LjVmfSIpCiAgICBwcmludChmIkNoZWNrcG9pbnQgc2F2ZWQgdG86IHtiZXN0X2NrcHRfcGF0aH0iKQogICAgcHJpbnQoZiJOb3JtYWxpemVyIHNhdmVkIHRvOiB7bW9kZWxzX2RpciAvICdub3JtYWxpemVyLm5weid9IikKICAgIHByaW50KCJOZXh0OiBweXRob24gLW0gdmlicmF0aW9uX2FkLmV2YWx1YXRlIC0tY29uZmlnIDxzYW1lIGNvbmZpZz4iKQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNvbmZpZyIsIGRlZmF1bHQ9ImNvbmZpZ3MvZGVmYXVsdC55YW1sIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoCiAgICAgICAgIi0tZm9yY2UtZGF0YSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsIGhlbHA9IlJlY29tcHV0ZSBjYWNoZWQgc3BlY3Ryb2dyYW1zIGV2ZW4gaWYgcHJlc2VudC4iCiAgICApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIGNvbmZpZyA9IGxvYWRfY29uZmlnKGFyZ3MuY29uZmlnKQogICAgdHJhaW4oY29uZmlnLCBjb25maWdfcGF0aD1hcmdzLmNvbmZpZywgZm9yY2VfZGF0YT1hcmdzLmZvcmNlX2RhdGEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'src/vibration_ad/evaluate.py': (
        'IiIiUGhhc2UgMzogdHVybiByZWNvbnN0cnVjdGlvbiBlcnJvciBpbnRvIGFub21hbHkgZGV0ZWN0aW9uLCBhbmQgc2NvcmUgaXQuCgpUaHJlc2hvbGQgc2VsZWN0aW9uIGlzIFVOU1VQRVJWSVNFRDogaXQncyBhIHBlcmNlbnRpbGUgb2YgdGhlIHJlY29uc3RydWN0aW9uCmVycm9yIG9uIHRoZSBWQUxJREFUSU9OIGhlYWx0aHkgc2V0IG9ubHkgKGNvbmZpZy5ldmFsdWF0aW9uLnRocmVzaG9sZF9wZXJjZW50aWxlLApkZWZhdWx0IDk5dGgpLiBObyBmYXVsdCBsYWJlbCBpcyB1c2VkIHRvIHBpY2sgaXQgLS0gdGhhdCdzIHRoZSBob25lc3QKc2ltdWxhdGlvbiBvZiAid2UgZG9uJ3QgaGF2ZSBsYWJlbHMsIHdlIG9ubHkga25vdyB3aGF0IGhlYWx0aHkgbG9va3MgbGlrZSIuCgpGYXVsdCBsYWJlbHMgZnJvbSB0aGUgQ1dSVSBmaWxlbmFtZXMgQVJFIHVzZWQgaGVyZSwgYnV0IG9ubHkgZG93bnN0cmVhbSBvZgp0aGF0IGNob2ljZSwgdG8gZ3JhZGUgdGhlIGRldGVjdG9yOiBST0MtQVVDLCBQUi1BVUMsIGNvbmZ1c2lvbiBtYXRyaXgsIGFuZApkZXRlY3Rpb24gcmF0ZSBicm9rZW4gZG93biBieSBmYXVsdCB0eXBlL3NldmVyaXR5LiBTZWUKbm90ZXMvMDRfZXZhbHVhdGlvbl9ub3Rlcy5tZCBmb3Igd2h5IHRoaXMgaXMgbGVnaXRpbWF0ZSBhbmQgd2hhdCBpdCBkb2Vzbid0CnByb3ZlIGFib3V0IGEgcmVhbCBkZXBsb3ltZW50LgoKVXNhZ2U6CiAgICBweXRob24gLW0gdmlicmF0aW9uX2FkLmV2YWx1YXRlIC0tY29uZmlnIGNvbmZpZ3MvZGVmYXVsdC55YW1sCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgKAogICAgYXZlcmFnZV9wcmVjaXNpb25fc2NvcmUsCiAgICBjb25mdXNpb25fbWF0cml4LAogICAgcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCwKICAgIHJvY19hdWNfc2NvcmUsCikKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyCgpmcm9tIHZpYnJhdGlvbl9hZCBpbXBvcnQgdml6CmZyb20gdmlicmF0aW9uX2FkLmNvbmZpZyBpbXBvcnQgQ29uZmlnLCBsb2FkX2NvbmZpZwpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLmRhdGFzZXQgaW1wb3J0IGJ1aWxkX3Jhd19kYXRhc2V0cwpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLnRvcmNoX2RhdGFzZXQgaW1wb3J0IFNwZWN0cm9ncmFtRGF0YXNldApmcm9tIHZpYnJhdGlvbl9hZC5mZWF0dXJlcy5zcGVjdHJvZ3JhbSBpbXBvcnQgU3BlY3Ryb2dyYW1Ob3JtYWxpemVyLCBleHBlY3RlZF9zaGFwZQpmcm9tIHZpYnJhdGlvbl9hZC5tb2RlbHMuYXV0b2VuY29kZXIgaW1wb3J0IENvbnZBdXRvZW5jb2RlciwgcGVyX3NhbXBsZV9tc2UsIHNlbGVjdF9kZXZpY2UKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBjb21wdXRlX2Vycm9yc19hbmRfcmVjb25zKG1vZGVsLCBYX25vcm06IG5wLm5kYXJyYXksIGRldmljZSwgYmF0Y2hfc2l6ZTogaW50ID0gNjQpOgogICAgbG9hZGVyID0gRGF0YUxvYWRlcihTcGVjdHJvZ3JhbURhdGFzZXQoWF9ub3JtKSwgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlKQogICAgZXJyb3JzLCByZWNvbnMgPSBbXSwgW10KICAgIG1vZGVsLmV2YWwoKQogICAgZm9yIGJhdGNoIGluIGxvYWRlcjoKICAgICAgICBiYXRjaCA9IGJhdGNoLnRvKGRldmljZSkKICAgICAgICByZWNvbiA9IG1vZGVsKGJhdGNoKQogICAgICAgIGVycm9ycy5hcHBlbmQocGVyX3NhbXBsZV9tc2UoYmF0Y2gsIHJlY29uKS5jcHUoKS5udW1weSgpKQogICAgICAgIHJlY29ucy5hcHBlbmQocmVjb24uY3B1KCkubnVtcHkoKSkKICAgIHJldHVybiBucC5jb25jYXRlbmF0ZShlcnJvcnMpLCBucC5jb25jYXRlbmF0ZShyZWNvbnMpWzosIDBdICAjIGRyb3AgY2hhbm5lbCBkaW0KCgpkZWYgZXZhbHVhdGUoY29uZmlnOiBDb25maWcpIC0+IGRpY3Q6CiAgICBtb2RlbHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLm1vZGVsc19kaXIpCiAgICByZXBvcnRzX2RpciA9IGNvbmZpZy5yZXNvbHZlKGNvbmZpZy5wYXRocy5yZXBvcnRzX2RpcikKICAgIGZpZ3VyZXNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLmZpZ3VyZXNfZGlyKQogICAgZm9yIGQgaW4gKHJlcG9ydHNfZGlyLCBmaWd1cmVzX2Rpcik6CiAgICAgICAgZC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgcHJpbnQoIkxvYWRpbmcgY2FjaGVkIGRhdGFzZXRzLi4uIikKICAgIF90cmFpbl9YLCB2YWxfWCwgX3ZhbF9tZXRhLCB0ZXN0X1gsIHRlc3RfbWV0YSA9IGJ1aWxkX3Jhd19kYXRhc2V0cyhjb25maWcpCgogICAgbm9ybWFsaXplciA9IFNwZWN0cm9ncmFtTm9ybWFsaXplci5sb2FkKG1vZGVsc19kaXIgLyAibm9ybWFsaXplci5ucHoiKQogICAgdmFsX1hfbm9ybSA9IG5vcm1hbGl6ZXIudHJhbnNmb3JtKHZhbF9YKQogICAgdGVzdF9YX25vcm0gPSBub3JtYWxpemVyLnRyYW5zZm9ybSh0ZXN0X1gpCgogICAgZXhwZWN0ZWQgPSBleHBlY3RlZF9zaGFwZShjb25maWcud2luZG93aW5nLCBjb25maWcuc3RmdCkKICAgIGRldmljZSA9IHNlbGVjdF9kZXZpY2UoY29uZmlnLnRyYWluLmRldmljZSkKICAgIG1vZGVsID0gQ29udkF1dG9lbmNvZGVyKAogICAgICAgIGlucHV0X3NoYXBlPWV4cGVjdGVkLAogICAgICAgIGJhc2VfY2hhbm5lbHM9Y29uZmlnLm1vZGVsLmJhc2VfY2hhbm5lbHMsCiAgICAgICAgbGF0ZW50X2RpbT1jb25maWcubW9kZWwubGF0ZW50X2RpbSwKICAgICkudG8oZGV2aWNlKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQobW9kZWxzX2RpciAvICJhdXRvZW5jb2Rlcl9iZXN0LnB0IiwgbWFwX2xvY2F0aW9uPWRldmljZSkpCgogICAgcHJpbnQoIkNvbXB1dGluZyByZWNvbnN0cnVjdGlvbiBlcnJvciBvbiB2YWxpZGF0aW9uIChoZWFsdGh5KSBzZXQuLi4iKQogICAgdmFsX2Vycm9ycywgXyA9IGNvbXB1dGVfZXJyb3JzX2FuZF9yZWNvbnMobW9kZWwsIHZhbF9YX25vcm0sIGRldmljZSkKICAgIHRocmVzaG9sZCA9IGZsb2F0KG5wLnBlcmNlbnRpbGUodmFsX2Vycm9ycywgY29uZmlnLmV2YWx1YXRpb24udGhyZXNob2xkX3BlcmNlbnRpbGUpKQogICAgcHJpbnQoZiJUaHJlc2hvbGQgKHtjb25maWcuZXZhbHVhdGlvbi50aHJlc2hvbGRfcGVyY2VudGlsZX10aCBwY3Qgb2YgdmFsIGhlYWx0aHkgZXJyb3IpOiB7dGhyZXNob2xkOi41Zn0iKQoKICAgIHByaW50KCJDb21wdXRpbmcgcmVjb25zdHJ1Y3Rpb24gZXJyb3Igb24gdGVzdCAoaGVhbHRoeSArIGZhdWx0eSkgc2V0Li4uIikKICAgIHRlc3RfZXJyb3JzLCB0ZXN0X3JlY29ucyA9IGNvbXB1dGVfZXJyb3JzX2FuZF9yZWNvbnMobW9kZWwsIHRlc3RfWF9ub3JtLCBkZXZpY2UpCiAgICB0ZXN0X21ldGEgPSB0ZXN0X21ldGEuY29weSgpCiAgICB0ZXN0X21ldGFbInJlY29uX2Vycm9yIl0gPSB0ZXN0X2Vycm9ycwogICAgdGVzdF9tZXRhWyJwcmVkaWN0ZWRfYW5vbWFseSJdID0gKHRlc3RfZXJyb3JzID4gdGhyZXNob2xkKS5hc3R5cGUoaW50KQoKICAgIHlfdHJ1ZSA9IHRlc3RfbWV0YVsibGFiZWwiXS50b19udW1weSgpCiAgICB5X3ByZWQgPSB0ZXN0X21ldGFbInByZWRpY3RlZF9hbm9tYWx5Il0udG9fbnVtcHkoKQoKICAgIHJvY19hdWMgPSByb2NfYXVjX3Njb3JlKHlfdHJ1ZSwgdGVzdF9lcnJvcnMpCiAgICBwcl9hdWMgPSBhdmVyYWdlX3ByZWNpc2lvbl9zY29yZSh5X3RydWUsIHRlc3RfZXJyb3JzKQogICAgcHJlY2lzaW9uLCByZWNhbGwsIGYxLCBfID0gcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCgKICAgICAgICB5X3RydWUsIHlfcHJlZCwgYXZlcmFnZT0iYmluYXJ5IiwgemVyb19kaXZpc2lvbj0wCiAgICApCiAgICB0biwgZnAsIGZuLCB0cCA9IGNvbmZ1c2lvbl9tYXRyaXgoeV90cnVlLCB5X3ByZWQpLnJhdmVsKCkKCiAgICBtZXRyaWNzID0gewogICAgICAgICJ0aHJlc2hvbGQiOiB0aHJlc2hvbGQsCiAgICAgICAgInRocmVzaG9sZF9wZXJjZW50aWxlIjogY29uZmlnLmV2YWx1YXRpb24udGhyZXNob2xkX3BlcmNlbnRpbGUsCiAgICAgICAgInJvY19hdWMiOiBmbG9hdChyb2NfYXVjKSwKICAgICAgICAicHJfYXVjIjogZmxvYXQocHJfYXVjKSwKICAgICAgICAicHJlY2lzaW9uIjogZmxvYXQocHJlY2lzaW9uKSwKICAgICAgICAicmVjYWxsIjogZmxvYXQocmVjYWxsKSwKICAgICAgICAiZjEiOiBmbG9hdChmMSksCiAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXgiOiB7InRuIjogaW50KHRuKSwgImZwIjogaW50KGZwKSwgImZuIjogaW50KGZuKSwgInRwIjogaW50KHRwKX0sCiAgICAgICAgIm5fdmFsX2hlYWx0aHkiOiBpbnQobGVuKHZhbF9lcnJvcnMpKSwKICAgICAgICAibl90ZXN0X2hlYWx0aHkiOiBpbnQoKHlfdHJ1ZSA9PSAwKS5zdW0oKSksCiAgICAgICAgIm5fdGVzdF9mYXVsdHkiOiBpbnQoKHlfdHJ1ZSA9PSAxKS5zdW0oKSksCiAgICB9CiAgICBwcmludChqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yKSkKCiAgICAjIERldGVjdGlvbiByYXRlIGJyb2tlbiBkb3duIGJ5IGZhdWx0IHR5cGUgLyBzZXZlcml0eSAoZmF1bHR5IHJvd3Mgb25seSkuCiAgICBmYXVsdHlfcm93cyA9IHRlc3RfbWV0YVt0ZXN0X21ldGEubGFiZWwgPT0gMV0KICAgIHNldmVyaXR5X3N1bW1hcnkgPSAoCiAgICAgICAgZmF1bHR5X3Jvd3MuZ3JvdXBieShbImZhdWx0X3R5cGUiLCAiZmF1bHRfc2l6ZV9taWxzIl0pCiAgICAgICAgLmFnZyhkZXRlY3Rpb25fcmF0ZT0oInByZWRpY3RlZF9hbm9tYWx5IiwgIm1lYW4iKSwgbj0oInByZWRpY3RlZF9hbm9tYWx5IiwgInNpemUiKSkKICAgICAgICAucmVzZXRfaW5kZXgoKQogICAgICAgIC5zb3J0X3ZhbHVlcyhbImZhdWx0X3R5cGUiLCAiZmF1bHRfc2l6ZV9taWxzIl0pCiAgICApCgogICAgIyAtLS0gc2F2ZSBhcnRpZmFjdHMgLS0tCiAgICB3aXRoIG9wZW4ocmVwb3J0c19kaXIgLyAiZXZhbHVhdGlvbl9tZXRyaWNzLmpzb24iLCAidyIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG1ldHJpY3MsIGYsIGluZGVudD0yKQogICAgdGVzdF9tZXRhLnRvX2NzdihyZXBvcnRzX2RpciAvICJ0ZXN0X3ByZWRpY3Rpb25zLmNzdiIsIGluZGV4PUZhbHNlKQogICAgc2V2ZXJpdHlfc3VtbWFyeS50b19jc3YocmVwb3J0c19kaXIgLyAiZGV0ZWN0aW9uX3JhdGVfYnlfc2V2ZXJpdHkuY3N2IiwgaW5kZXg9RmFsc2UpCgogICAgaGVhbHRoeV9lcnJvcnMgPSB0ZXN0X2Vycm9yc1t5X3RydWUgPT0gMF0KICAgIGZhdWx0eV9lcnJvcnMgPSB0ZXN0X2Vycm9yc1t5X3RydWUgPT0gMV0KICAgIHZpei5wbG90X2Vycm9yX2hpc3RvZ3JhbShoZWFsdGh5X2Vycm9ycywgZmF1bHR5X2Vycm9ycywgdGhyZXNob2xkLCBmaWd1cmVzX2RpciAvICJlcnJvcl9oaXN0b2dyYW0ucG5nIikKICAgIHZpei5wbG90X3JvY19jdXJ2ZSh5X3RydWUsIHRlc3RfZXJyb3JzLCByb2NfYXVjLCBmaWd1cmVzX2RpciAvICJyb2NfY3VydmUucG5nIikKICAgIHZpei5wbG90X2RldGVjdGlvbl9yYXRlX2J5X3NldmVyaXR5KHNldmVyaXR5X3N1bW1hcnksIGZpZ3VyZXNfZGlyIC8gImRldGVjdGlvbl9yYXRlX2J5X3NldmVyaXR5LnBuZyIpCgogICAgIyBBIGZldyBxdWFsaXRhdGl2ZSBleGFtcGxlczogYmVzdC1yZWNvbnN0cnVjdGVkIGhlYWx0aHkgd2luZG93LCBhbmQgdGhlCiAgICAjIGZhdWx0eSB3aW5kb3cgd2l0aCB0aGUgaGlnaGVzdCBlcnJvciAoY2xlYXJlc3QgIndoYXQgd2VudCB3cm9uZyIpLgogICAgZXhhbXBsZXMgPSBbXQogICAgaWR4X2hlYWx0aHlfYmVzdCA9IG5wLmFyZ21pbihucC53aGVyZSh5X3RydWUgPT0gMCwgdGVzdF9lcnJvcnMsIG5wLmluZikpCiAgICBpZHhfZmF1bHR5X3dvcnN0ID0gbnAuYXJnbWF4KG5wLndoZXJlKHlfdHJ1ZSA9PSAxLCB0ZXN0X2Vycm9ycywgLW5wLmluZikpCiAgICBmb3IgbGFiZWwsIGlkeCBpbiBbKCJoZWFsdGh5IChsb3cgZXJyb3IpIiwgaWR4X2hlYWx0aHlfYmVzdCksICgiZmF1bHR5IChoaWdoIGVycm9yKSIsIGlkeF9mYXVsdHlfd29yc3QpXToKICAgICAgICBleGFtcGxlcy5hcHBlbmQoKGxhYmVsLCB0ZXN0X1hfbm9ybVtpZHhdLCB0ZXN0X3JlY29uc1tpZHhdKSkKICAgIHZpei5wbG90X3JlY29uc3RydWN0aW9uX2V4YW1wbGVzKGV4YW1wbGVzLCBmaWd1cmVzX2RpciAvICJyZWNvbnN0cnVjdGlvbl9leGFtcGxlcy5wbmciKQoKICAgIHByaW50KGYiXG5TYXZlZCBtZXRyaWNzL3Bsb3RzIHRvIHtyZXBvcnRzX2Rpcn0gYW5kIHtmaWd1cmVzX2Rpcn0iKQogICAgcmV0dXJuIG1ldHJpY3MKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jb25maWciLCBkZWZhdWx0PSJjb25maWdzL2RlZmF1bHQueWFtbCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgY29uZmlnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBldmFsdWF0ZShjb25maWcpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'src/vibration_ad/infer.py': (
        'IiIiUnVuIHRoZSB0cmFpbmVkIGRldGVjdG9yIG9uIG9uZSBzaWduYWwgZmlsZSBhbmQgdmlzdWFsaXplIHRoZSByZXN1bHQuCgpUaGlzIGlzIHRoZSAiZGVwbG95bWVudCIgZW50cnlwb2ludDogcG9pbnQgaXQgYXQgYW55IENXUlUgLm5weiBmaWxlIGFuZCBpdApyZXByb2R1Y2VzIHRoZSB3aG9sZSBwaXBlbGluZSAod2luZG93IC0+IFNURlQgLT4gbm9ybWFsaXplIC0+IHJlY29uc3RydWN0IC0+CmVycm9yKSBlbmQgdG8gZW5kIG9uIHRoYXQgb25lIGZpbGUsIHRoZW4gcGxvdHMgcmVjb25zdHJ1Y3Rpb24gZXJyb3Igb3Zlcgp0aW1lIHdpdGggdGhlIGNhbGlicmF0ZWQgdGhyZXNob2xkIC0tIHRoZSAiVmlzdWFsaXppbmcgQXV0b2VuY29kZXIKUmVjb25zdHJ1Y3Rpb24gRXJyb3IiIHZpZXcgZnJvbSB0aGUgb3JpZ2luYWwgcHJvamVjdCBwbGFuLgoKUmVxdWlyZXMgYGV2YWx1YXRlLnB5YCB0byBoYXZlIGJlZW4gcnVuIGZpcnN0IChpdCByZXVzZXMgdGhlIHRocmVzaG9sZApjYWxpYnJhdGVkIHRoZXJlLCByYXRoZXIgdGhhbiByZWNvbXB1dGluZyBpdCwgc28gdGhlIGZsYWdnaW5nIGJlaGF2aW9yIGhlcmUKYWx3YXlzIG1hdGNoZXMgdGhlIHJlcG9ydGVkIGV2YWx1YXRpb24gbWV0cmljcykuCgpVc2FnZToKICAgIHB5dGhvbiAtbSB2aWJyYXRpb25fYWQuaW5mZXIgLS1jb25maWcgY29uZmlncy9kZWZhdWx0LnlhbWwgXFwKICAgICAgICAtLWlucHV0IGRhdGEvcmF3LzE3OTdfSVJfMjFfREUxMi5ucHoKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCgpmcm9tIHZpYnJhdGlvbl9hZC5jb25maWcgaW1wb3J0IENvbmZpZywgbG9hZF9jb25maWcKZnJvbSB2aWJyYXRpb25fYWQuZGF0YS5jYXRhbG9nIGltcG9ydCBwYXJzZV9maWxlbmFtZQpmcm9tIHZpYnJhdGlvbl9hZC5kYXRhLmRhdGFzZXQgaW1wb3J0IGxvYWRfc2lnbmFsCmZyb20gdmlicmF0aW9uX2FkLmRhdGEud2luZG93cyBpbXBvcnQgbWFrZV93aW5kb3dzCmZyb20gdmlicmF0aW9uX2FkLmV2YWx1YXRlIGltcG9ydCBjb21wdXRlX2Vycm9yc19hbmRfcmVjb25zCmZyb20gdmlicmF0aW9uX2FkLmZlYXR1cmVzLnNwZWN0cm9ncmFtIGltcG9ydCAoCiAgICBTcGVjdHJvZ3JhbU5vcm1hbGl6ZXIsCiAgICBleHBlY3RlZF9zaGFwZSwKICAgIHdpbmRvd3NfdG9fbG9nX3NwZWN0cm9ncmFtcywKKQpmcm9tIHZpYnJhdGlvbl9hZC5tb2RlbHMuYXV0b2VuY29kZXIgaW1wb3J0IENvbnZBdXRvZW5jb2Rlciwgc2VsZWN0X2RldmljZQpmcm9tIHZpYnJhdGlvbl9hZCBpbXBvcnQgdml6CgoKZGVmIGluZmVyKGNvbmZpZzogQ29uZmlnLCBpbnB1dF9wYXRoOiBzdHIsIHN0cmlkZTogaW50IHwgTm9uZSA9IE5vbmUpIC0+IHBkLkRhdGFGcmFtZToKICAgIG1vZGVsc19kaXIgPSBjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMubW9kZWxzX2RpcikKICAgIHJlcG9ydHNfZGlyID0gY29uZmlnLnJlc29sdmUoY29uZmlnLnBhdGhzLnJlcG9ydHNfZGlyKQogICAgZmlndXJlc19kaXIgPSBjb25maWcucmVzb2x2ZShjb25maWcucGF0aHMuZmlndXJlc19kaXIpCiAgICBmaWd1cmVzX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgbWV0cmljc19wYXRoID0gcmVwb3J0c19kaXIgLyAiZXZhbHVhdGlvbl9tZXRyaWNzLmpzb24iCiAgICBpZiBub3QgbWV0cmljc19wYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKAogICAgICAgICAgICBmInttZXRyaWNzX3BhdGh9IG5vdCBmb3VuZC4gUnVuIGBweXRob24gLW0gdmlicmF0aW9uX2FkLmV2YWx1YXRlYCBmaXJzdCAiCiAgICAgICAgICAgICJzbyBpbmZlci5weSBjYW4gcmV1c2UgdGhlIGNhbGlicmF0ZWQgdGhyZXNob2xkLiIKICAgICAgICApCiAgICB0aHJlc2hvbGQgPSBqc29uLmxvYWRzKG1ldHJpY3NfcGF0aC5yZWFkX3RleHQoKSlbInRocmVzaG9sZCJdCgogICAgc3RyaWRlID0gc3RyaWRlIG9yIGNvbmZpZy53aW5kb3dpbmcuZXZhbF9zdHJpZGUKICAgIHNpZ25hbCA9IGxvYWRfc2lnbmFsKGlucHV0X3BhdGgsIGNvbmZpZy5kYXRhLnNlbnNvcikKICAgIHdpbmRvd3MgPSBtYWtlX3dpbmRvd3Moc2lnbmFsLCBjb25maWcud2luZG93aW5nLnNlZ21lbnRfbGVuZ3RoLCBzdHJpZGUpCiAgICBpZiBsZW4od2luZG93cykgPT0gMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICBmIlNpZ25hbCBoYXMge2xlbihzaWduYWwpfSBzYW1wbGVzLCBzaG9ydGVyIHRoYW4gc2VnbWVudF9sZW5ndGg9IgogICAgICAgICAgICBmIntjb25maWcud2luZG93aW5nLnNlZ21lbnRfbGVuZ3RofTsgY2FuJ3QgZm9ybSBldmVuIG9uZSB3aW5kb3cuIgogICAgICAgICkKCiAgICBzcGVjdHJvZ3JhbXMgPSB3aW5kb3dzX3RvX2xvZ19zcGVjdHJvZ3JhbXMod2luZG93cywgY29uZmlnLmRhdGEuZnNfaHosIGNvbmZpZy5zdGZ0KQogICAgbm9ybWFsaXplciA9IFNwZWN0cm9ncmFtTm9ybWFsaXplci5sb2FkKG1vZGVsc19kaXIgLyAibm9ybWFsaXplci5ucHoiKQogICAgc3BlY3Ryb2dyYW1zX25vcm0gPSBub3JtYWxpemVyLnRyYW5zZm9ybShzcGVjdHJvZ3JhbXMpCgogICAgZXhwZWN0ZWQgPSBleHBlY3RlZF9zaGFwZShjb25maWcud2luZG93aW5nLCBjb25maWcuc3RmdCkKICAgIGRldmljZSA9IHNlbGVjdF9kZXZpY2UoY29uZmlnLnRyYWluLmRldmljZSkKICAgIG1vZGVsID0gQ29udkF1dG9lbmNvZGVyKAogICAgICAgIGlucHV0X3NoYXBlPWV4cGVjdGVkLAogICAgICAgIGJhc2VfY2hhbm5lbHM9Y29uZmlnLm1vZGVsLmJhc2VfY2hhbm5lbHMsCiAgICAgICAgbGF0ZW50X2RpbT1jb25maWcubW9kZWwubGF0ZW50X2RpbSwKICAgICkudG8oZGV2aWNlKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHRvcmNoLmxvYWQobW9kZWxzX2RpciAvICJhdXRvZW5jb2Rlcl9iZXN0LnB0IiwgbWFwX2xvY2F0aW9uPWRldmljZSkpCgogICAgZXJyb3JzLCBfcmVjb25zID0gY29tcHV0ZV9lcnJvcnNfYW5kX3JlY29ucyhtb2RlbCwgc3BlY3Ryb2dyYW1zX25vcm0sIGRldmljZSkKICAgIGZsYWdnZWQgPSBlcnJvcnMgPiB0aHJlc2hvbGQKCiAgICByZXN1bHQgPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgewogICAgICAgICAgICAid2luZG93X2luZGV4IjogbnAuYXJhbmdlKGxlbihlcnJvcnMpKSwKICAgICAgICAgICAgInN0YXJ0X3NhbXBsZSI6IG5wLmFyYW5nZShsZW4oZXJyb3JzKSkgKiBzdHJpZGUsCiAgICAgICAgICAgICJyZWNvbl9lcnJvciI6IGVycm9ycywKICAgICAgICAgICAgInByZWRpY3RlZF9hbm9tYWx5IjogZmxhZ2dlZC5hc3R5cGUoaW50KSwKICAgICAgICB9CiAgICApCgogICAgZm5hbWUgPSBpbnB1dF9wYXRoLnJzcGxpdCgiLyIsIDEpWy0xXQogICAgdHJ5OgogICAgICAgIG1ldGEgPSBwYXJzZV9maWxlbmFtZShmbmFtZSkKICAgICAgICByZXN1bHRbInRydWVfZmF1bHRfdHlwZSJdID0gbWV0YVsiZmF1bHRfdHlwZSJdCiAgICAgICAgcmVzdWx0WyJ0cnVlX2xhYmVsIl0gPSBtZXRhWyJsYWJlbCJdCiAgICAgICAgZ3JvdW5kX3RydXRoX25vdGUgPSBmIiAodHJ1ZSBsYWJlbDoge21ldGFbJ2ZhdWx0X3R5cGUnXX0sIHNpemU9e21ldGFbJ2ZhdWx0X3NpemVfbWlscyddfW1pbCkiCiAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICBncm91bmRfdHJ1dGhfbm90ZSA9ICIgKGZpbGVuYW1lIG5vdCBpbiBDV1JVIG5hbWluZyBjb252ZW50aW9uIC0tIG5vIGdyb3VuZCB0cnV0aCB0byBjb21wYXJlKSIKCiAgICBvdXRfY3N2ID0gcmVwb3J0c19kaXIgLyBmImluZmVyX3tmbmFtZS5yZXBsYWNlKCcubnB6JywgJycpfS5jc3YiCiAgICByZXN1bHQudG9fY3N2KG91dF9jc3YsIGluZGV4PUZhbHNlKQoKICAgIG91dF9wbmcgPSBmaWd1cmVzX2RpciAvIGYiaW5mZXJfe2ZuYW1lLnJlcGxhY2UoJy5ucHonLCAnJyl9X3RpbWVsaW5lLnBuZyIKICAgIHZpei5wbG90X2Vycm9yX3RpbWVsaW5lKGVycm9ycywgdGhyZXNob2xkLCBvdXRfcG5nLCB0aXRsZT1mIntmbmFtZX17Z3JvdW5kX3RydXRoX25vdGV9IikKCiAgICBuX2ZsYWdnZWQgPSBpbnQoZmxhZ2dlZC5zdW0oKSkKICAgIHByaW50KGYie2ZuYW1lfToge25fZmxhZ2dlZH0ve2xlbihlcnJvcnMpfSB3aW5kb3dzIGZsYWdnZWQgYXMgYW5vbWFsb3VzICh7MTAwICogbl9mbGFnZ2VkIC8gbGVuKGVycm9ycyk6LjFmfSUpIikKICAgIHByaW50KGYiVGltZWxpbmUgcGxvdDoge291dF9wbmd9IikKICAgIHByaW50KGYiUGVyLXdpbmRvdyBDU1Y6IHtvdXRfY3N2fSIpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZmlnIiwgZGVmYXVsdD0iY29uZmlncy9kZWZhdWx0LnlhbWwiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1pbnB1dCIsIHJlcXVpcmVkPVRydWUsIGhlbHA9IlBhdGggdG8gYSBDV1JVIC5ucHogc2lnbmFsIGZpbGUiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1zdHJpZGUiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLCBoZWxwPSJXaW5kb3cgc3RyaWRlIGluIHNhbXBsZXMgKGRlZmF1bHQ6IGNvbmZpZyBldmFsX3N0cmlkZSkiCiAgICApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgY29uZmlnID0gbG9hZF9jb25maWcoYXJncy5jb25maWcpCiAgICBpbmZlcihjb25maWcsIGFyZ3MuaW5wdXQsIHN0cmlkZT1hcmdzLnN0cmlkZSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
    ),
    'src/vibration_ad/viz.py': (
        'IiIiTWF0cGxvdGxpYiBwbG90dGluZyBoZWxwZXJzIHNoYXJlZCBieSBldmFsdWF0ZS5weSBhbmQgaW5mZXIucHkuCgpFdmVyeSBmdW5jdGlvbiBzYXZlcyBhIFBORyB0byB0aGUgZ2l2ZW4gcGF0aCBhbmQgcmV0dXJucyBpdDsgbm9uZSBjYWxsCnBsdC5zaG93KCkgc2luY2UgdGhpcyBpcyBtZWFudCB0byBydW4gaGVhZGxlc3MgKFZTIENvZGUgLyBDTEkgLyBDSSkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBtYXRwbG90bGliCgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19jdXJ2ZQoKCmRlZiBwbG90X2Vycm9yX2hpc3RvZ3JhbSgKICAgIGhlYWx0aHlfZXJyb3JzOiBucC5uZGFycmF5LCBmYXVsdHlfZXJyb3JzOiBucC5uZGFycmF5LCB0aHJlc2hvbGQ6IGZsb2F0LCBzYXZlX3BhdGg6IFBhdGgKKSAtPiBQYXRoOgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg4LCA1KSkKICAgIGJpbnMgPSBucC5saW5zcGFjZSgKICAgICAgICAwLCBtYXgoaGVhbHRoeV9lcnJvcnMubWF4KCksIGZhdWx0eV9lcnJvcnMubWF4KCkpICogMS4wNSwgNjAKICAgICkKICAgIGF4Lmhpc3QoaGVhbHRoeV9lcnJvcnMsIGJpbnM9YmlucywgYWxwaGE9MC42LCBsYWJlbD1mImhlYWx0aHkgKG49e2xlbihoZWFsdGh5X2Vycm9ycyl9KSIsIGNvbG9yPSIjM2I4MmY2IikKICAgIGF4Lmhpc3QoZmF1bHR5X2Vycm9ycywgYmlucz1iaW5zLCBhbHBoYT0wLjYsIGxhYmVsPWYiZmF1bHR5IChuPXtsZW4oZmF1bHR5X2Vycm9ycyl9KSIsIGNvbG9yPSIjZWY0NDQ0IikKICAgIGF4LmF4dmxpbmUodGhyZXNob2xkLCBjb2xvcj0iYmxhY2siLCBsaW5lc3R5bGU9Ii0tIiwgbGluZXdpZHRoPTEuNSwgbGFiZWw9ZiJ0aHJlc2hvbGQ9e3RocmVzaG9sZDouNGZ9IikKICAgIGF4LnNldF94bGFiZWwoIlJlY29uc3RydWN0aW9uIGVycm9yIChNU0UpIikKICAgIGF4LnNldF95bGFiZWwoIldpbmRvdyBjb3VudCIpCiAgICBheC5zZXRfdGl0bGUoIlJlY29uc3RydWN0aW9uIGVycm9yOiBoZWFsdGh5IHZzLiBmYXVsdHkiKQogICAgYXgubGVnZW5kKCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcoc2F2ZV9wYXRoLCBkcGk9MTUwKQogICAgcGx0LmNsb3NlKGZpZykKICAgIHJldHVybiBzYXZlX3BhdGgKCgpkZWYgcGxvdF9yb2NfY3VydmUoeV90cnVlOiBucC5uZGFycmF5LCBzY29yZXM6IG5wLm5kYXJyYXksIGF1YzogZmxvYXQsIHNhdmVfcGF0aDogUGF0aCkgLT4gUGF0aDoKICAgIGZwciwgdHByLCBfID0gcm9jX2N1cnZlKHlfdHJ1ZSwgc2NvcmVzKQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg2LCA2KSkKICAgIGF4LnBsb3QoZnByLCB0cHIsIGNvbG9yPSIjM2I4MmY2IiwgbGFiZWw9ZiJBVUMgPSB7YXVjOi4zZn0iKQogICAgYXgucGxvdChbMCwgMV0sIFswLCAxXSwgY29sb3I9ImdyYXkiLCBsaW5lc3R5bGU9Ii0tIiwgbGluZXdpZHRoPTEpCiAgICBheC5zZXRfeGxhYmVsKCJGYWxzZSBwb3NpdGl2ZSByYXRlIikKICAgIGF4LnNldF95bGFiZWwoIlRydWUgcG9zaXRpdmUgcmF0ZSIpCiAgICBheC5zZXRfdGl0bGUoIlJPQyBjdXJ2ZSAodW5zdXBlcnZpc2VkIGFub21hbHkgc2NvcmUgdnMuIHRydWUgZmF1bHQgbGFiZWwpIikKICAgIGF4LmxlZ2VuZCgpCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIGZpZy5zYXZlZmlnKHNhdmVfcGF0aCwgZHBpPTE1MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gc2F2ZV9wYXRoCgoKZGVmIHBsb3RfZGV0ZWN0aW9uX3JhdGVfYnlfc2V2ZXJpdHkoc3VtbWFyeTogInBkLkRhdGFGcmFtZSIsIHNhdmVfcGF0aDogUGF0aCkgLT4gUGF0aDoKICAgICIiInN1bW1hcnk6IGNvbHVtbnMgZmF1bHRfdHlwZSwgZmF1bHRfc2l6ZV9taWxzLCBkZXRlY3Rpb25fcmF0ZSwgbi4iIiIKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQgICMgbG9jYWwgaW1wb3J0OiBrZWVwIHRvcC1sZXZlbCBtb2R1bGUgbnVtcHkvc2tsZWFybiBvbmx5CgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSg5LCA1KSkKICAgIGxhYmVscyA9IFtmIntyLmZhdWx0X3R5cGV9XG57ci5mYXVsdF9zaXplX21pbHN9bWlsIiBmb3IgciBpbiBzdW1tYXJ5Lml0ZXJ0dXBsZXMoKV0KICAgIGF4LmJhcihsYWJlbHMsIHN1bW1hcnlbImRldGVjdGlvbl9yYXRlIl0sIGNvbG9yPSIjMjJjNTVlIikKICAgIGF4LnNldF95bGltKDAsIDEuMDUpCiAgICBheC5zZXRfeWxhYmVsKCJEZXRlY3Rpb24gcmF0ZSBhdCB0aHJlc2hvbGQiKQogICAgYXguc2V0X3RpdGxlKCJEZXRlY3Rpb24gcmF0ZSBieSBmYXVsdCB0eXBlIGFuZCBzZXZlcml0eSIpCiAgICBheC50aWNrX3BhcmFtcyhheGlzPSJ4Iiwgcm90YXRpb249NDUpCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIGZpZy5zYXZlZmlnKHNhdmVfcGF0aCwgZHBpPTE1MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gc2F2ZV9wYXRoCgoKZGVmIHBsb3RfcmVjb25zdHJ1Y3Rpb25fZXhhbXBsZXMoCiAgICBleGFtcGxlczogbGlzdFt0dXBsZVtzdHIsIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldXSwgc2F2ZV9wYXRoOiBQYXRoCikgLT4gUGF0aDoKICAgICIiImV4YW1wbGVzOiBsaXN0IG9mICh0aXRsZSwgb3JpZ2luYWxfc3BlY3Ryb2dyYW0sIHJlY29uc3RydWN0ZWRfc3BlY3Ryb2dyYW0pLAogICAgZWFjaCAoRiwgVCkuIFJlbmRlcnMgb3JpZ2luYWwgLyByZWNvbnN0cnVjdGlvbiAvIHxkaWZmfCBjb2x1bW5zIHBlciByb3cuCiAgICAiIiIKICAgIG4gPSBsZW4oZXhhbXBsZXMpCiAgICBmaWcsIGF4ZXMgPSBwbHQuc3VicGxvdHMobiwgMywgZmlnc2l6ZT0oOSwgMyAqIG4pKQogICAgaWYgbiA9PSAxOgogICAgICAgIGF4ZXMgPSBheGVzW05vbmUsIDpdCiAgICBmb3IgaSwgKHRpdGxlLCBvcmlnLCByZWNvbikgaW4gZW51bWVyYXRlKGV4YW1wbGVzKToKICAgICAgICBkaWZmID0gbnAuYWJzKG9yaWcgLSByZWNvbikKICAgICAgICBmb3IgaiwgKGFyciwgc3VidGl0bGUpIGluIGVudW1lcmF0ZSgKICAgICAgICAgICAgWyhvcmlnLCAiaW5wdXQiKSwgKHJlY29uLCAicmVjb25zdHJ1Y3Rpb24iKSwgKGRpZmYsICJ8ZGlmZmVyZW5jZXwiKV0KICAgICAgICApOgogICAgICAgICAgICBheCA9IGF4ZXNbaSwgal0KICAgICAgICAgICAgaW0gPSBheC5pbXNob3coYXJyLCBhc3BlY3Q9ImF1dG8iLCBvcmlnaW49Imxvd2VyIiwgY21hcD0ibWFnbWEiKQogICAgICAgICAgICBheC5zZXRfdGl0bGUoZiJ7dGl0bGV9IC0ge3N1YnRpdGxlfSIgaWYgaiA9PSAwIGVsc2Ugc3VidGl0bGUsIGZvbnRzaXplPTkpCiAgICAgICAgICAgIGF4LnNldF94dGlja3MoW10pCiAgICAgICAgICAgIGF4LnNldF95dGlja3MoW10pCiAgICAgICAgICAgIGZpZy5jb2xvcmJhcihpbSwgYXg9YXgsIGZyYWN0aW9uPTAuMDQ2LCBwYWQ9MC4wNCkKICAgIGZpZy50aWdodF9sYXlvdXQoKQogICAgZmlnLnNhdmVmaWcoc2F2ZV9wYXRoLCBkcGk9MTUwKQogICAgcGx0LmNsb3NlKGZpZykKICAgIHJldHVybiBzYXZlX3BhdGgKCgpkZWYgcGxvdF9lcnJvcl90aW1lbGluZSgKICAgIGVycm9yczogbnAubmRhcnJheSwgdGhyZXNob2xkOiBmbG9hdCwgc2F2ZV9wYXRoOiBQYXRoLCB0aXRsZTogc3RyID0gIlJlY29uc3RydWN0aW9uIGVycm9yIG92ZXIgdGltZSIKKSAtPiBQYXRoOgogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhmaWdzaXplPSgxMCwgNCkpCiAgICB4ID0gbnAuYXJhbmdlKGxlbihlcnJvcnMpKQogICAgZmxhZ2dlZCA9IGVycm9ycyA+IHRocmVzaG9sZAogICAgYXgucGxvdCh4LCBlcnJvcnMsIGNvbG9yPSIjM2I4MmY2IiwgbGluZXdpZHRoPTEsIGxhYmVsPSJyZWNvbnN0cnVjdGlvbiBlcnJvciIpCiAgICBheC5zY2F0dGVyKHhbZmxhZ2dlZF0sIGVycm9yc1tmbGFnZ2VkXSwgY29sb3I9IiNlZjQ0NDQiLCBzPTIwLCB6b3JkZXI9MywgbGFiZWw9ImZsYWdnZWQgYW5vbWFseSIpCiAgICBheC5heGhsaW5lKHRocmVzaG9sZCwgY29sb3I9ImJsYWNrIiwgbGluZXN0eWxlPSItLSIsIGxpbmV3aWR0aD0xLjIsIGxhYmVsPWYidGhyZXNob2xkPXt0aHJlc2hvbGQ6LjRmfSIpCiAgICBheC5zZXRfeGxhYmVsKCJXaW5kb3cgaW5kZXggKHRpbWUgb3JkZXIpIikKICAgIGF4LnNldF95bGFiZWwoIlJlY29uc3RydWN0aW9uIGVycm9yIChNU0UpIikKICAgIGF4LnNldF90aXRsZSh0aXRsZSkKICAgIGF4LmxlZ2VuZCgpCiAgICBmaWcudGlnaHRfbGF5b3V0KCkKICAgIGZpZy5zYXZlZmlnKHNhdmVfcGF0aCwgZHBpPTE1MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gc2F2ZV9wYXRoCg=='
    ),
}
for relpath, b64 in SRC_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(SRC_FILES)} source files under {WORKDIR}/src')

In [ ]:
CONFIG_FILES = {
    'configs/default.yaml': (
        'IyBEZWZhdWx0IGNvbmZpZ3VyYXRpb24gZm9yIHRoZSB2aWJyYXRpb24gYW5vbWFseSBkZXRlY3Rpb24gTVZQLgojIEV2ZXJ5IHN0YWdlIChkYXRhIHByZXAsIHRyYWluaW5nLCBldmFsdWF0aW9uLCBpbmZlcmVuY2UpIHJlYWRzIHRoaXMKIyBvbmUgZmlsZSBzbyBhIGZ1bGwgcnVuIGlzIHJlcHJvZHVjaWJsZSBmcm9tIGEgc2luZ2xlIHNvdXJjZSBvZiB0cnV0aC4KIyBPdmVycmlkZSBieSBjb3B5aW5nIHRoaXMgZmlsZSBhbmQgcG9pbnRpbmcgc2NyaXB0cyBhdCB0aGUgY29weSwgZS5nLjoKIyAgIHB5dGhvbiAtbSB2aWJyYXRpb25fYWQudHJhaW4gLS1jb25maWcgY29uZmlncy9teV9leHBlcmltZW50LnlhbWwKCnNlZWQ6IDQyCgpwYXRoczoKICByYXdfZGlyOiBkYXRhL3JhdwogIHByb2Nlc3NlZF9kaXI6IGRhdGEvcHJvY2Vzc2VkCiAgbW9kZWxzX2Rpcjogb3V0cHV0cy9tb2RlbHMKICBmaWd1cmVzX2Rpcjogb3V0cHV0cy9maWd1cmVzCiAgcmVwb3J0c19kaXI6IG91dHB1dHMvcmVwb3J0cwoKIyAtLS0gRGF0YSBzZWxlY3Rpb24gLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgQ1dSVSBEcml2ZS1FbmQgYWNjZWxlcm9tZXRlciBzaWduYWxzLCAxMiBrSHogc2FtcGxpbmcgcmF0ZSAoREUxMiBmaWxlcyksCiMgcGFpcmVkIHdpdGggdGhlIDEyIGtIeiBOb3JtYWwtYmFzZWxpbmUgZmlsZSBmb3IgdGhlIHNhbWUgUlBNL2xvYWQgZmFtaWx5LgojIFNlZSBub3Rlcy8wMV9kYXRhc2V0X25vdGVzLm1kIGZvciB3aHkgdGhpcyBzdWJzZXQgd2FzIGNob3Nlbi4KZGF0YToKICBzZW5zb3I6IERFICAgICAgICAgICAgIyBhcnJheSBrZXkgaW5zaWRlIGVhY2ggLm5weiBmaWxlOiBERSwgRkUsIG9yIEJBCiAgZnNfaHo6IDEyMDAwICAgICAgICAgICAjIHNhbXBsaW5nIHJhdGUgb2YgdGhlIERFMTIgLyBOb3JtYWwgZmlsZXMKICAjIFJQTSBmYW1pbGllcyB1c2VkIGFzIEhFQUxUSFkgVFJBSU5JTkcgZGF0YSAobmV2ZXIgc2VlIGEgZmF1bHQgbGFiZWwpLgogIHRyYWluX3JwbXM6IFsxNzMwLCAxNzUwLCAxNzcyXQogICMgUlBNIGZhbWlseSBoZWxkIG91dCBlbnRpcmVseSBmcm9tIHRyYWluaW5nOiBpdHMgTm9ybWFsIGZpbGUgc3VwcGxpZXMKICAjIGhlbGQtb3V0IGhlYWx0aHkgdmFsL3Rlc3Qgd2luZG93cywgYW5kIGl0cyBmYXVsdCBmaWxlcyBhcmUgcHVyZSB0ZXN0IGRhdGEuCiAgaG9sZG91dF9ycG06IDE3OTcKICAjIEZhdWx0IGZhbWlsaWVzIHRvIGluY2x1ZGUgd2hlbiBidWlsZGluZyB0aGUgKGV2YWwtb25seSkgZmF1bHR5IHNldC4KICAjIE9SQDMvT1JANi9PUkAxMiA9IG91dGVyLXJhY2UgZmF1bHQgYXQgMy82LzEyIG8nY2xvY2sgcmVsYXRpdmUgdG8gbG9hZCB6b25lLgogIGZhdWx0X3R5cGVzOiBbSVIsIEIsICJPUkAzIiwgIk9SQDYiLCAiT1JAMTIiXQogICMgUlBNIGZhbWlsaWVzIHRvIHB1bGwgRkFVTFRZIGZpbGVzIGZyb20uIEZhdWx0eSBmaWxlcyBhcmUgbmV2ZXIgdXNlZCBmb3IKICAjIHRyYWluaW5nIHJlZ2FyZGxlc3Mgb2YgUlBNLCBzbyBhbGwgZm91ciBhcmUgZmFpciBnYW1lIGZvciBldmFsdWF0aW9uCiAgIyAodGhpcyBhbHNvIHRlc3RzIHdoZXRoZXIgdGhlIGRldGVjdG9yIGdlbmVyYWxpemVzIGFjcm9zcyBsb2FkIGNvbmRpdGlvbnMpLgogIGZhdWx0X3JwbXM6IFsxNzMwLCAxNzUwLCAxNzcyLCAxNzk3XQoKIyAtLS0gV2luZG93aW5nIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBMb25nIHJhdyBzaWduYWxzIGFyZSBjdXQgaW50byBmaXhlZC1sZW5ndGggb3ZlcmxhcHBpbmcgd2luZG93cyBiZWZvcmUKIyBhbnkgc3BlY3Ryb2dyYW0gaXMgY29tcHV0ZWQuIFN0cmlkZSBjb250cm9scyB3aW5kb3cgZGVuc2l0eSAvIG92ZXJsYXAuCndpbmRvd2luZzoKICBzZWdtZW50X2xlbmd0aDogODM4NCAgICAjIHNhbXBsZXMgcGVyIHdpbmRvdyAofjAuNzBzIEAgMTJrSHosIH4yMSBzaGFmdCByZXZzIEAxNzk3cnBtKQogIHRyYWluX3N0cmlkZTogNTEyICAgICAgICAjIGRlbnNlIG92ZXJsYXAgZm9yIGhlYWx0aHkgVFJBSU4gd2luZG93cyAobW9yZSB0cmFpbmluZyBkYXRhKQogIGV2YWxfc3RyaWRlOiAyMDk2ICAgICAgICAgIyBzcGFyc2VyIG92ZXJsYXAgZm9yIHZhbC90ZXN0IHdpbmRvd3MgKGxlc3MgcmVkdW5kYW5jeSkKCiMgLS0tIFNURlQgZmVhdHVyZSBleHRyYWN0aW9uIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIENob3NlbiB0byBtYWtlIGVhY2ggd2luZG93IG1hcCB0byBFWEFDVExZIGEgMTI4eDEyOCBsb2ctbWFnbml0dWRlIGltYWdlCiMgKHNlZSBmZWF0dXJlcy9zcGVjdHJvZ3JhbS5weTpleHBlY3RlZF9zaGFwZSBmb3IgdGhlIGRlcml2YXRpb24pLCBzbyB0aGUKIyBjb252IGF1dG9lbmNvZGVyIGFyY2hpdGVjdHVyZSBiZWxvdyBjYW4gYXNzdW1lIGEgZml4ZWQgaW5wdXQgc2l6ZS4Kc3RmdDoKICBucGVyc2VnOiAyNTYKICBob3BfbGVuZ3RoOiA2NCAgICAgICAgICAjIG5vdmVybGFwID0gbnBlcnNlZyAtIGhvcF9sZW5ndGggPSAxOTIKICBsb2dfZXBzaWxvbjogMS4wZS0xMCAgICAgIyBhZGRlZCBiZWZvcmUgbG9nIHRvIGF2b2lkIGxvZygwKQoKIyAtLS0gTW9kZWwgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQptb2RlbDoKICBsYXRlbnRfZGltOiAxMjgKICBiYXNlX2NoYW5uZWxzOiAxNiAgICAgICAgIyBjaGFubmVsIGNvdW50IG9mIGZpcnN0IGNvbnYgbGF5ZXI7IGRvdWJsZXMgZWFjaCBsYXllcgoKIyAtLS0gVHJhaW5pbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQp0cmFpbjoKICBiYXRjaF9zaXplOiAzMgogIGVwb2NoczogNjAKICBsZWFybmluZ19yYXRlOiAxLjBlLTMKICB3ZWlnaHRfZGVjYXk6IDEuMGUtNQogIGVhcmx5X3N0b3BwaW5nX3BhdGllbmNlOiAxMAogIHZhbF9mcmFjdGlvbl9vZl9ob2xkb3V0OiAwLjUgICAjIGZyYWN0aW9uIG9mIGhvbGRvdXQtUlBNIGhlYWx0aHkgd2luZG93cyB1c2VkIGZvciB2YWwgKHJlc3QgLT4gdGVzdCkKICBkZXZpY2U6IGF1dG8gICAgICAgICAgICAgICMgYXV0byB8IGNwdSB8IG1wcyB8IGN1ZGEKCiMgLS0tIEV2YWx1YXRpb24gLyB0aHJlc2hvbGRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmV2YWx1YXRpb246CiAgIyBVbnN1cGVydmlzZWQgdGhyZXNob2xkOiBwZXJjZW50aWxlIG9mIHJlY29uc3RydWN0aW9uIGVycm9yIG9uIHRoZQogICMgVkFMSURBVElPTiBoZWFsdGh5IHNldCAobGFiZWxzIGFyZSBOT1QgdXNlZCB0byBwaWNrIHRoaXMpLiBTZWUKICAjIG5vdGVzLzA0X2V2YWx1YXRpb25fbm90ZXMubWQgZm9yIHdoeSB0aGlzIGlzIHRoZSBkZXBsb3ltZW50LWhvbmVzdCBjaG9pY2UuCiAgdGhyZXNob2xkX3BlcmNlbnRpbGU6IDk5LjAK'
    ),
    'data/raw/MANIFEST.txt': (
        'RGF0YS8xNzMwIFJQTS8xNzMwX0JfMTRfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX0JfMjFfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX0JfMjhfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX0JfN19ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfSVJfMTRfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX0lSXzIxX0RFMTIubnB6CkRhdGEvMTczMCBSUE0vMTczMF9JUl8yOF9ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfSVJfN19ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfTm9ybWFsLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfT1JAMTJfMjFfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX09SQDEyXzdfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX09SQDNfMjFfREUxMi5ucHoKRGF0YS8xNzMwIFJQTS8xNzMwX09SQDNfN19ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfT1JANl8xNF9ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfT1JANl8yMV9ERTEyLm5wegpEYXRhLzE3MzAgUlBNLzE3MzBfT1JANl83X0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9CXzE0X0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9CXzIxX0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9CXzI4X0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9CXzdfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX0lSXzE0X0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9JUl8yMV9ERTEyLm5wegpEYXRhLzE3NTAgUlBNLzE3NTBfSVJfMjhfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX0lSXzdfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX05vcm1hbC5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX09SQDEyXzIxX0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9PUkAxMl83X0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9PUkAzXzIxX0RFMTIubnB6CkRhdGEvMTc1MCBSUE0vMTc1MF9PUkAzXzdfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX09SQDZfMTRfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX09SQDZfMjFfREUxMi5ucHoKRGF0YS8xNzUwIFJQTS8xNzUwX09SQDZfN19ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfQl8xNF9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfQl8yMV9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfQl8yOF9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfQl83X0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9JUl8xNF9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfSVJfMjFfREUxMi5ucHoKRGF0YS8xNzcyIFJQTS8xNzcyX0lSXzI4X0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9JUl83X0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9Ob3JtYWwubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9PUkAxMl8yMV9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfT1JAMTJfN19ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfT1JAM18yMV9ERTEyLm5wegpEYXRhLzE3NzIgUlBNLzE3NzJfT1JAM183X0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9PUkA2XzE0X0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9PUkA2XzIxX0RFMTIubnB6CkRhdGEvMTc3MiBSUE0vMTc3Ml9PUkA2XzdfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X0JfMTRfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X0JfMjFfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X0JfMjhfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X0JfN19ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfSVJfMTRfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X0lSXzIxX0RFMTIubnB6CkRhdGEvMTc5NyBSUE0vMTc5N19JUl8yOF9ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfSVJfN19ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfTm9ybWFsLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfT1JAMTJfMjFfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X09SQDEyXzdfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X09SQDNfMjFfREUxMi5ucHoKRGF0YS8xNzk3IFJQTS8xNzk3X09SQDNfN19ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfT1JANl8xNF9ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfT1JANl8yMV9ERTEyLm5wegpEYXRhLzE3OTcgUlBNLzE3OTdfT1JANl83X0RFMTIubnB6Cg=='
    ),
}
for relpath, b64 in CONFIG_FILES.items():
    p = WORKDIR / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print(f'wrote {len(CONFIG_FILES)} config/manifest files')

## 2. Environment setup + confirm PyTorch

The `!python -m ...` calls below run in a **separate shell subprocess** -- `sys.path.insert()` in this process doesn't reach them, and plain `python` on the PATH is not guaranteed to be the same interpreter as this notebook kernel (it can resolve to a bare system Python without torch/numpy/etc installed). So: set `PYTHONPATH` via `os.environ` (subprocesses inherit that) and always invoke `{sys.executable}` explicitly (guaranteed to be *this* kernel's interpreter) rather than bare `python`.

In [ ]:
import sys, os, subprocess
sys.path.insert(0, str(WORKDIR / 'src'))
os.environ['PYTHONPATH'] = str(WORKDIR / 'src')
os.chdir(WORKDIR)
PY = sys.executable
try:
    import yaml  # noqa -- already on Kaggle's base image in practice
except ImportError:
    # best-effort: this needs internet, which may not be available
    # (see the markdown above) -- if it's already installed, this
    # branch never runs and nothing depends on internet access.
    subprocess.run([PY, '-m', 'pip', 'install', '-q', 'pyyaml'])

import torch
print('python:', PY)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available(), '(CPU-only run, expected)')

In [ ]:
def run(*args):
    # check=True -> raises (and stops the notebook) on a non-zero exit,
    # instead of silently continuing to the next cell like `!` would.
    subprocess.run([PY, *args], check=True)

## 3. Convert the CWRU dataset from the attached input (.mat -> .npz)
`esraakhaled299/cwru-data` mirrors the same 64 files (12kHz Drive-End fault files + Normal baseline, all 4 RPM/load conditions) as `data/raw/MANIFEST.txt`, just as `.mat` instead of `.npz` and organized into folders instead of RPM-prefixed filenames. Original CWRU `.mat` files use a numeric-ID key prefix that varies per file (e.g. `X100_DE_time`, `X285_DE_time`) -- match by suffix (`_DE_time`) rather than hardcoding the ID. Mount path confirmed by a throwaway diagnostic kernel first (`notebooks/inspect_kernel/`) rather than assumed.

In [ ]:
import scipy.io, re
import numpy as np

# Mount path for attached input datasets has proven inconsistent across
# kernels in practice (confirmed .../datasets/<owner>/<slug>/... via a
# diagnostic kernel, then a real run still didn't find it there) --
# search for the known folder name instead of hardcoding a path.
_candidates = list(pathlib.Path('/kaggle/input').rglob('CWRU_organized'))
if not _candidates:
    print('CWRU_organized not found under /kaggle/input. Contents:')
    for p in sorted(pathlib.Path('/kaggle/input').rglob('*')):
        print(' ', p)
    raise FileNotFoundError(
        "cwru-data dataset not found under /kaggle/input -- is "
        "'esraakhaled299/cwru-data' attached to this kernel? (see "
        "kernel-metadata.json's dataset_sources)"
    )
MAT_ROOT = _candidates[0]
print(f'MAT_ROOT = {MAT_ROOT}')
RAW_DIR = WORKDIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
LOAD_TO_RPM = {0: 1797, 1: 1772, 2: 1750, 3: 1730}

BALL_IR_RE = re.compile(r'^(B|IR)(\d{3})_(\d)\.mat$')
OR_RE = re.compile(r'^OR(\d{3})@(\d+)_(\d)\.mat$')
NORMAL_RE = re.compile(r'^Normal_(\d)\.mat$')

def de_signal(mat_path):
    d = scipy.io.loadmat(str(mat_path))
    key = next(k for k in d if k.endswith('_DE_time'))
    return d[key].ravel()

n = 0
for mat_path in sorted((MAT_ROOT / '12k_DE').rglob('*.mat')):
    fname = mat_path.name
    m = BALL_IR_RE.match(fname)
    if m:
        fault, size, load = m.group(1), int(m.group(2)), int(m.group(3))
        out_name = f'{LOAD_TO_RPM[load]}_{fault}_{size}_DE12.npz'
    else:
        m = OR_RE.match(fname)
        if not m:
            print(f'skip (unrecognized): {mat_path}')
            continue
        size, pos, load = int(m.group(1)), m.group(2), int(m.group(3))
        out_name = f'{LOAD_TO_RPM[load]}_OR@{pos}_{size}_DE12.npz'
    np.savez(RAW_DIR / out_name, DE=de_signal(mat_path))
    n += 1

for mat_path in sorted((MAT_ROOT / 'Normal').rglob('*.mat')):
    m = NORMAL_RE.match(mat_path.name)
    if not m:
        print(f'skip (unrecognized): {mat_path}')
        continue
    rpm = LOAD_TO_RPM[int(m.group(1))]
    np.savez(RAW_DIR / f'{rpm}_Normal.npz', DE=de_signal(mat_path))
    n += 1

print(f'converted {n} files into {RAW_DIR}')
assert n == 64, f'expected 64 files (matching data/raw/MANIFEST.txt), got {n}'

## 4. Train the autoencoder
Healthy-only spectrograms, early-stopped on held-out healthy validation loss. Runs on CPU (see device print above) -- this is a small model (~a few hundred thousand params) on ~2800 training windows, so it's slow-ish but not impractical on CPU.

In [ ]:
run('-m', 'vibration_ad.train', '--config', 'configs/default.yaml')

## 5. Evaluate: reconstruction error -> anomaly detection, scored against CWRU's true fault labels

In [ ]:
run('-m', 'vibration_ad.evaluate', '--config', 'configs/default.yaml')

In [ ]:
import json
print(json.dumps(json.loads((WORKDIR / 'outputs/reports/evaluation_metrics.json').read_text()), indent=2))

## 6. Demo inference on one healthy and one faulty file

In [ ]:
run('-m', 'vibration_ad.infer', '--config', 'configs/default.yaml', '--input', 'data/raw/1797_IR_21_DE12.npz')
run('-m', 'vibration_ad.infer', '--config', 'configs/default.yaml', '--input', 'data/raw/1797_Normal.npz')

## 7. Look at the results

In [ ]:
from IPython.display import Image, display
figdir = WORKDIR / 'outputs' / 'figures'
for name in ['error_histogram.png', 'roc_curve.png', 'detection_rate_by_severity.png',
             'reconstruction_examples.png', 'infer_1797_IR_21_DE12_timeline.png',
             'infer_1797_Normal_timeline.png']:
    path = figdir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))